# MICA 2026: Reproducible BUSSAM Prompt-Robustness Pipeline

This Colab-GPU notebook turns the existing BUSSAM reproduction into an auditable study. It provides split auditing, deterministic multi-seed execution, prompt-robustness testing, a matched U-Net baseline, per-case metrics, bootstrap confidence intervals, figures, and exportable evidence.

**Research claim supported after completion:** robustness and reproducibility analysis of an existing BUSSAM implementation. This notebook does **not** claim BUSSAM as a new model.

Validated against the supplied `train.py` and `test.py`: both hard-code seeds in the upstream code, so the instrumentation cell replaces those assignments with environment-controlled seeds while preserving backups.


## 1. Connect Drive and configure paths

Set `PROJECT_ROOT` to the existing `BUSSAM-main` directory. Training switches are intentionally off on first run.

In [1]:
from pathlib import Path
import os, sys, json, shutil, hashlib, re, subprocess, time, random
import numpy as np

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = next(
    Path("/kaggle/input").rglob("BUSSAM-main")
)

#print("Found BUSSAM project:", PROJECT_ROOT)

STUDY_ROOT = Path(
    "/kaggle/working/MICA2026_BUSSAM_STUDY"
)

# DO NOT create PROJECT_ROOT
# /kaggle/input is read-only

# Create only writable output directories
for p in [
    STUDY_ROOT,
    STUDY_ROOT / "logs",
    STUDY_ROOT / "checkpoints",
    STUDY_ROOT / "per_case",
    STUDY_ROOT / "tables",
    STUDY_ROOT / "figures"
]:
    p.mkdir(parents=True, exist_ok=True)


# ============================================================
# SETTINGS
# ============================================================

SEEDS = [42, 52, 62]

RUN_BUSSAM_TRAINING = True
RUN_UNET_BASELINE = True
RUN_BUSSAM_ROBUSTNESS = True

UNET_EPOCHS = 100
IMAGE_SIZE = 256
BOOTSTRAPS = 2000


# ============================================================
# CHECK PROJECT
# ============================================================

assert PROJECT_ROOT.exists(), (
    f"BUSSAM project not found: {PROJECT_ROOT}"
)

assert (PROJECT_ROOT / "datasets").exists(), (
    f"Datasets folder not found: {PROJECT_ROOT / 'datasets'}"
)

assert (PROJECT_ROOT / "models").exists(), (
    f"Models folder not found: {PROJECT_ROOT / 'models'}"
)

# Move into project
os.chdir(PROJECT_ROOT)

print("Found BUSSAM project: BUSSAM-main")
print("PROJECT_ROOT: BUSSAM-main")
print("STUDY_ROOT: MICA2026_BUSSAM_STUDY")
print("Project: BUSSAM-main")
print("Study outputs: MICA2026_BUSSAM_STUDY")
print("Current directory: BUSSAM-main")

Found BUSSAM project: BUSSAM-main
PROJECT_ROOT: BUSSAM-main
STUDY_ROOT: MICA2026_BUSSAM_STUDY
Project: BUSSAM-main
Study outputs: MICA2026_BUSSAM_STUDY
Current directory: BUSSAM-main


## 2. Environment and repository audit

In [2]:
import platform, torch, pandas as pd
required = ['train.py','test.py','utils/config.py','utils/data_us.py','utils/evaluation.py','utils/metrics.py']
missing = [x for x in required if not (PROJECT_ROOT/x).exists()]
if missing:
    raise FileNotFoundError('Missing required BUSSAM files: ' + ', '.join(missing))

env = {
    'generated_utc': pd.Timestamp.utcnow().isoformat(),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seeds': SEEDS,
}
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU in Colab.'
(STUDY_ROOT/'environment.json').write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))


{
  "generated_utc": "2026-08-21T18:04:07.524883+00:00",
  "python": "3.12.13",
  "torch": "2.10.0+cu128",
  "cuda_available": true,
  "gpu": "Tesla T4",
  "seeds": [
    42,
    52,
    62
  ]
}


In [3]:
from pathlib import Path

# PROJECT_ROOT is already detected automatically in Cell 1

DATASET_ROOT = PROJECT_ROOT / "datasets"
SPLIT_ROOT = DATASET_ROOT / "MainPatient"

print("PROJECT_ROOT: BUSSAM-main")
print("DATASET_ROOT: datasets")
print("SPLIT_ROOT: datasets/MainPatient")

print("\nChecking dataset folders/files...\n")

print("datasets exists:", DATASET_ROOT.exists())
print("MainPatient exists:", SPLIT_ROOT.exists())

for split in ["train", "val", "test"]:
    path = SPLIT_ROOT / f"BUSI_{split}.txt"
    print(split, ":", path.exists(), f"datasets/MainPatient/BUSI_{split}.txt")

PROJECT_ROOT: BUSSAM-main
DATASET_ROOT: datasets
SPLIT_ROOT: datasets/MainPatient

Checking dataset folders/files...

datasets exists: True
MainPatient exists: True
train : True datasets/MainPatient/BUSI_train.txt
val : True datasets/MainPatient/BUSI_val.txt
test : True datasets/MainPatient/BUSI_test.txt


## 3. Split integrity, class counts, and duplicate audit

The cell fails if the same identifier appears in more than one split. It also checks image/label existence and exact duplicate image bytes across splits.

In [4]:
import pandas as pd, cv2

def locate_split(name):
    candidates = [SPLIT_ROOT/f'BUSI_{name}.txt', DATASET_ROOT/'BUSI'/'MainPatient'/f'BUSI_{name}.txt']
    for p in candidates:
        if p.exists(): return p
    raise FileNotFoundError(f'Cannot locate BUSI_{name}.txt')

def parse_identifier(raw):
    parts = raw.strip().split('/')
    if len(parts) < 3: raise ValueError(f'Unexpected split identifier: {raw}')
    class_id, subpath, stem = parts[0], parts[-2], parts[-1]
    return class_id, subpath, stem

rows=[]
for split in ['train','val','test']:
    sf=locate_split(split)
    for line in sf.read_text().splitlines():
        if not line.strip(): continue
        class_id, subpath, stem=parse_identifier(line)
        img=DATASET_ROOT/subpath/'img'/f'{stem}.png'
        mask=DATASET_ROOT/subpath/'label'/f'{stem}.png'
        rows.append({'split':split,'raw_id':line.strip(),'class_id':class_id,'subpath':subpath,
                     'stem':stem,'image':str(img),'mask':str(mask),
                     'image_exists':img.exists(),'mask_exists':mask.exists()})
manifest=pd.DataFrame(rows)
assert manifest.image_exists.all() and manifest.mask_exists.all(), 'Some images or masks are missing.'

overlap=manifest.groupby('raw_id').split.nunique()
assert not (overlap>1).any(), 'Identifier overlap across splits detected.'

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
manifest['image_sha256']=[sha256(p) for p in manifest.image]
manifest['mask_sha256']=[sha256(p) for p in manifest['mask']]
dup=manifest.groupby('image_sha256').split.nunique()
duplicates = dup[dup > 1]

print("Number of duplicate images:", len(duplicates))

if len(duplicates) > 0:
    duplicate_rows = manifest[
        manifest["image_sha256"].isin(duplicates.index)
    ].copy()

    duplicate_rows["image_name"] = duplicate_rows["image"].apply(
        lambda x: Path(x).name
    )
    duplicate_rows["mask_name"] = duplicate_rows["mask"].apply(
        lambda x: Path(x).name
    )

    display(
        duplicate_rows[
            ["split", "image_name", "mask_name"]
        ]
    )
def mask_pixels(path):
    a=cv2.imread(path,0)
    return int((a>0).sum()), int(a.size)
mp=[mask_pixels(p) for p in manifest['mask']]
manifest['lesion_pixels']=[x[0] for x in mp]
manifest['total_pixels']=[x[1] for x in mp]
manifest['lesion_present']=manifest.lesion_pixels>0
manifest['category']=manifest.stem.str.extract(r'^(benign|malignant|normal)',expand=False).fillna('unknown')
manifest.to_csv(STUDY_ROOT/'split_manifest_audited.csv',index=False)
summary=manifest.groupby(['split','category','lesion_present']).size().rename('n').reset_index()
summary.to_csv(STUDY_ROOT/'tables'/'split_summary.csv',index=False)
display(summary)
print('Total unique samples:',len(manifest))
print('Audit passed.')


Number of duplicate images: 1


,split,image_name,mask_name
337,train,malignant_malignant_145.png,malignant_malignant_145.png
597,val,benign_benign_433.png,benign_benign_433.png


,split,category,lesion_present,n
0,test,benign,True,75
1,test,malignant,True,25
2,test,normal,False,21
3,train,benign,True,304
4,train,malignant,True,147
5,train,normal,False,98
6,val,benign,True,58
7,val,malignant,True,38
8,val,normal,False,14


Total unique samples: 780
Audit passed.


In [5]:
# ============================================================
# REMOVE CONFLICTING CROSS-SPLIT DUPLICATE
# ============================================================

import pandas as pd

# Find image hashes appearing in more than one split
image_split_counts = (
    manifest.groupby("image_sha256")["split"]
    .nunique()
)

duplicate_hashes = set(
    image_split_counts[image_split_counts > 1].index
)

print("Cross-split duplicate image hashes:", len(duplicate_hashes))

# Show what will be removed
conflicting_rows = manifest[
    manifest["image_sha256"].isin(duplicate_hashes)
].copy()

print("\nSamples to exclude:")

display(
    conflicting_rows[
        ["split", "raw_id", "category"]
    ]
)

# Remove BOTH copies
clean_manifest = manifest[
    ~manifest["image_sha256"].isin(duplicate_hashes)
].copy()

print("\nOriginal samples :", len(manifest))
print("Removed samples  :", len(manifest) - len(clean_manifest))
print("Clean samples    :", len(clean_manifest))

# Verify no image occurs across multiple splits
remaining_duplicates = (
    clean_manifest.groupby("image_sha256")["split"]
    .nunique()
)

assert not (
    remaining_duplicates > 1
).any(), "Cross-split duplicates still exist!"

# Verify identifiers also remain unique
remaining_identifier_overlap = (
    clean_manifest.groupby("raw_id")["split"]
    .nunique()
)

assert not (
    remaining_identifier_overlap > 1
).any(), "Identifier overlap still exists!"

print("\n✅ Clean split audit passed.")

Cross-split duplicate image hashes: 1

Samples to exclude:


,split,raw_id,category
337,train,1/BUSI/malignant_malignant_145,malignant
597,val,1/BUSI/benign_benign_433,benign



Original samples : 780
Removed samples  : 2
Clean samples    : 778

✅ Clean split audit passed.


In [6]:
# Save clean audited manifest

clean_manifest_path = (
    STUDY_ROOT / "split_manifest_clean.csv"
)

clean_manifest.to_csv(
    clean_manifest_path,
    index=False
)

print("Saved:", clean_manifest_path)

# Summary
clean_summary = (
    clean_manifest
    .groupby(["split", "category", "lesion_present"])
    .size()
    .rename("n")
    .reset_index()
)

display(clean_summary)

clean_summary.to_csv(
    STUDY_ROOT / "tables" / "split_summary_clean.csv",
    index=False
)

print("\nClean samples:", len(clean_manifest))

Saved: /kaggle/working/MICA2026_BUSSAM_STUDY/split_manifest_clean.csv


,split,category,lesion_present,n
0,test,benign,True,75
1,test,malignant,True,25
2,test,normal,False,21
3,train,benign,True,304
4,train,malignant,True,146
5,train,normal,False,98
6,val,benign,True,57
7,val,malignant,True,38
8,val,normal,False,14



Clean samples: 778


In [7]:
from pathlib import Path

# Use the project already detected earlier
DATASET_ROOT = PROJECT_ROOT

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

images = [
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

print("Total number of images:", len(images))

Total number of images: 1561


In [8]:
# BUSSAM uses text files to define train/val/test splits

SPLIT_ROOT = PROJECT_ROOT / "datasets" / "MainPatient"

for split in ["train", "val", "test"]:

    split_path = SPLIT_ROOT / f"BUSI_{split}.txt"

    if split_path.exists():

        lines = [
            line.strip()
            for line in split_path.read_text().splitlines()
            if line.strip()
        ]

        print(split, ":", len(lines), "samples")

    else:
        print(split, ": split file not found")

train : 549 samples
val : 110 samples
test : 121 samples


## 4. Install reversible research instrumentation

This appends two clearly marked blocks to the official code. Original files are backed up once. The first block adds deterministic centre, random-lesion, boundary, and displaced-point prompts controlled through environment variables. The second block records per-case test metrics. Re-running the cell is idempotent.

In [9]:
from pathlib import Path
import shutil

# Automatically locate the original BUSSAM project
SOURCE_ROOT = next(
    p for p in Path("/kaggle/input").rglob("BUSSAM-main")
    if p.is_dir()
)

# Writable copy
PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")

if not SOURCE_ROOT.exists():
    raise FileNotFoundError("Source BUSSAM project not found")

if not PROJECT_ROOT.exists():
    shutil.copytree(SOURCE_ROOT, PROJECT_ROOT)

In [10]:
dup = manifest.groupby("image_sha256")["split"].nunique()

duplicates = dup[dup > 1]

print(duplicates)

image_sha256
058e7538fa53b2bd8b87500ff47721c84ec442dc964d9de21bb271a8651138a8    2
Name: split, dtype: int64


In [11]:
from pathlib import Path

def backup_once(path):
    backup=path.with_suffix(path.suffix+'.mica_original')
    if not backup.exists(): shutil.copy2(path,backup)
    return backup

data_file=PROJECT_ROOT/'utils'/'data_us.py'
metrics_file=PROJECT_ROOT/'utils'/'metrics.py'
train_file=PROJECT_ROOT/'train.py'
test_file=PROJECT_ROOT/'test.py'
backup_once(data_file); backup_once(metrics_file); backup_once(train_file); backup_once(test_file)

prompt_patch=r'''
# === MICA2026_PROMPT_ROBUSTNESS_PATCH ===
import os as _mica_os, hashlib as _mica_hashlib
_mica_original_fixed_click = fixed_click

def _mica_rng(mask):
    seed=int(_mica_os.environ.get('MICA_PROMPT_SEED','42'))
    digest=int(_mica_hashlib.sha256(np.ascontiguousarray(mask).tobytes()).hexdigest()[:8],16)
    return np.random.default_rng(seed ^ digest)

def fixed_click(mask, class_id=1):
    mode=_mica_os.environ.get('MICA_PROMPT_MODE','center')
    coords=np.argwhere(mask==class_id)
    if len(coords)==0:
        # Negative prompt for lesion-absent images.
        h,w=mask.shape[:2]; return np.array([[[w//2,h//2]]]).reshape(1,2), [0]
    rng=_mica_rng(mask)
    if mode=='random_lesion':
        y,x=coords[rng.integers(len(coords))]
    elif mode=='boundary':
        binary=(mask==class_id).astype(np.uint8)
        edge=binary-cv2.erode(binary,np.ones((3,3),np.uint8),iterations=1)
        b=np.argwhere(edge>0); y,x=b[rng.integers(len(b))]
    else:
        y,x=coords[len(coords)//2]
    if mode.startswith('displaced_'):
        frac=float(mode.split('_')[1])/100.0
        h,w=mask.shape[:2]; angle=rng.uniform(0,2*np.pi)
        x=int(round(x+frac*w*np.cos(angle))); y=int(round(y+frac*h*np.sin(angle)))
        x=np.clip(x,0,w-1); y=np.clip(y,0,h-1)
    return np.array([[x,y]]), [1]
# === END_MICA2026_PROMPT_ROBUSTNESS_PATCH ===
'''

metric_patch=r'''
# === MICA2026_PER_CASE_METRIC_PATCH ===
import os as _mica_os, csv as _mica_csv, numpy as _mica_np
_mica_dice_original=dice_coefficient
_mica_ses_original=sespiou_coefficient2
_mica_hd_original=get_HD
_mica_pending={}
_mica_case_index=0
def _mica_float(x):
    try: return float(x.detach().cpu().item())
    except Exception: return float(_mica_np.asarray(x).reshape(-1)[0])
def dice_coefficient(pred,gt,smooth=1e-5):
    out=_mica_dice_original(pred,gt,smooth); _mica_pending['dice']=_mica_float(out); return out
def sespiou_coefficient2(pred,gt,all=False,smooth=1e-5):
    out=_mica_ses_original(pred,gt,all,smooth)
    vals=[_mica_float(v) for v in out]
    if all: keys=['sensitivity','specificity','iou','accuracy','f1','precision','recall']
    else: keys=['iou','accuracy','sensitivity','specificity']
    _mica_pending.update(dict(zip(keys,vals))); return out
def get_HD(x,y,p=2):
    global _mica_case_index
    out=_mica_hd_original(x,y,p); path=_mica_os.environ.get('MICA_CASE_LOG')
    if path:
        row={'case_index':_mica_case_index,**_mica_pending,'hausdorff':_mica_float(out)}
        exists=_mica_os.path.exists(path)
        with open(path,'a',newline='') as f:
            w=_mica_csv.DictWriter(f,fieldnames=list(row));
            if not exists: w.writeheader()
            w.writerow(row)
        _mica_case_index+=1; _mica_pending.clear()
    return out
# === END_MICA2026_PER_CASE_METRIC_PATCH ===
'''

for path,marker,patch in [(data_file,'MICA2026_PROMPT_ROBUSTNESS_PATCH',prompt_patch),
                          (metrics_file,'MICA2026_PER_CASE_METRIC_PATCH',metric_patch)]:
    text=path.read_text()
    if marker not in text:
        path.write_text(text+'\n'+patch+'\n')
        print('Patched:',path)
    else: print('Already patched:',path)


# The supplied train.py and test.py overwrite external seeds with constants
# 1234 and 300. Replace only that assignment and retain one-time backups.
for path in [train_file, test_file]:
    text = path.read_text()
    changed = re.sub(
        r'(?m)^(\s*)seed_value\s*=\s*\d+\s*# the number of seed',
        lambda m: m.group(1) + "seed_value = int(os.environ.get('MICA_SEED', '1234'))  # MICA-controlled seed",
        text,
    )
    if changed == text and 'MICA-controlled seed' not in text:
        raise RuntimeError(f'Could not locate hard-coded seed in {path}')
    path.write_text(changed)
    print('Seed control verified:', path)


Patched: /kaggle/working/BUSSAM-main/utils/data_us.py
Patched: /kaggle/working/BUSSAM-main/utils/metrics.py
Seed control verified: /kaggle/working/BUSSAM-main/train.py
Seed control verified: /kaggle/working/BUSSAM-main/test.py


## 5. Deterministic BUSSAM launcher

The launcher seeds Python, NumPy, and PyTorch before executing the repository script. It preserves each run log and copies the newly generated epoch-100 checkpoint into the study directory.

In [12]:
print("Current directory: BUSSAM-main")

Current directory: BUSSAM-main


In [13]:
print("Project contents:")

!ls

Project contents:
BUSSAM.png   datasets  outputs	  requirements.txt  train.py
checkpoints  models    README.md  test.py	    utils


In [14]:
!pip install hausdorff

  Preparing metadata (setup.py) ... done
  Created wheel for hausdorff: filename=hausdorff-0.2.6-py3-none-any.whl size=15244 sha256=dbd2d1c9f57d6a3e1ef6e1a7df6ff46630db7b8697c34c89e029515a021c3612
  Stored in directory: /root/.cache/pip/wheels/31/0d/1c/f5d96bf935d71576a8e65dd31e6518586108a9c35a1de5e6b9
Successfully built hausdorff


In [15]:
import hausdorff
print("hausdorff installed successfully")

hausdorff installed successfully


In [16]:
import os
print(os.getcwd())

/kaggle/input/datasets/sridharmanukonda/breast-cancer/BUSSAM-main


In [17]:
from batchgenerators.utilities.file_and_folder_operations import *

ModuleNotFoundError: No module named 'batchgenerators'

In [ ]:
!pip install -q batchgenerators

In [ ]:
import batchgenerators
print("batchgenerators installed successfully")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
train_file = PROJECT_ROOT / "train.py"

text = train_file.read_text()

# GPU ni disable chestunna line remove
text = text.replace(
    "os.environ['CUDA_VISIBLE_DEVICES'] = ''",
    ""
)

# CPU ni CUDA ga marchu
text = text.replace(
    'opt.device = "cpu"',
    'opt.device = "cuda:0"'
)

text = text.replace(
    'opt.cuda = "off"',
    'opt.cuda = "on"'
)

train_file.write_text(text)

print("✅ GPU enabled in train.py")

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available!"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
from pathlib import Path
import shutil

# Automatically locate the original BUSSAM project
SOURCE_ROOT = next(
    p for p in Path("/kaggle/input").rglob("BUSSAM-main")
    if p.is_dir()
)

# Writable working copy
PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        "Original BUSSAM project not found"
    )

if PROJECT_ROOT.exists():
    print("Writable BUSSAM project already exists.")
else:
    print("Creating writable BUSSAM project...")
    shutil.copytree(SOURCE_ROOT, PROJECT_ROOT)

print("PROJECT_ROOT: BUSSAM-main")
print("Exists:", PROJECT_ROOT.exists())

In [ ]:
from pathlib import Path
import torch

PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

assert PROJECT_ROOT.exists(), f"Project not found: {PROJECT_ROOT}"
assert (PROJECT_ROOT / "train.py").exists(), "train.py not found"

assert torch.cuda.is_available(), "❌ GPU is OFF"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("✅ Ready to train on GPU")

In [ ]:
launcher=STUDY_ROOT/'seeded_launcher.py'
launcher.write_text(r'''import os,sys,runpy,random
sys.path.insert(0, os.getcwd())
import numpy as np, torch
seed=int(os.environ['MICA_SEED'])
os.environ['PYTHONHASHSEED']=str(seed)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
sys.argv=[os.environ['MICA_SCRIPT']]
runpy.run_path(os.environ['MICA_SCRIPT'],run_name='__main__')
''')

def run_logged(cmd,env,log_path):
    with open(log_path,'w') as log:
        proc=subprocess.Popen(cmd,cwd=PROJECT_ROOT,env={**os.environ,**env},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end=''); log.write(line); log.flush()
        rc=proc.wait()
    if rc: raise RuntimeError(f'Command failed ({rc}); see {log_path}')

def newest_checkpoint(since=0):
    files=[p for p in (PROJECT_ROOT/'outputs').rglob('BUSSAM_100.pth') if p.stat().st_mtime>=since]
    if not files: raise FileNotFoundError('No newly generated BUSSAM_100.pth found.')
    return max(files,key=lambda p:p.stat().st_mtime)

if RUN_BUSSAM_TRAINING:
    for seed in SEEDS:
        dest=STUDY_ROOT/'checkpoints'/f'BUSSAM_seed{seed}.pth'
        if dest.exists(): print('Checkpoint exists; skipped:',dest); continue
        started=time.time()
        run_logged([sys.executable,str(launcher)],{'MICA_SEED':str(seed),'MICA_SCRIPT':'train.py'},STUDY_ROOT/'logs'/f'train_seed{seed}.log')
        src=newest_checkpoint(started)
        shutil.copy2(src,dest)
        print('Saved:',dest)
else:
    print('Training disabled. Set RUN_BUSSAM_TRAINING=True after audit approval.')


In [ ]:
import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

# Paths
PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

os.chdir(PROJECT_ROOT)

# Check GPU
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


# ---------- run_logged ----------
def run_logged(cmd, env, log_path):
    with open(log_path, "w") as log:
        proc = subprocess.Popen(
            cmd,
            cwd=PROJECT_ROOT,
            env={**os.environ, **env},
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()

        rc = proc.wait()

        if rc != 0:
            raise RuntimeError(
                f"Training failed with exit code {rc}. Check: {log_path}"
            )


# ---------- find newly generated checkpoint ----------
def newest_checkpoint(since=0):
    files = [
        p for p in (PROJECT_ROOT / "outputs").rglob("BUSSAM_100.pth")
        if p.stat().st_mtime >= since
    ]

    if not files:
        raise FileNotFoundError(
            "No newly generated BUSSAM_100.pth found."
        )

    return max(files, key=lambda p: p.stat().st_mtime)


# ---------- Seed 62 only ----------
seed = 62

dest = STUDY_ROOT / "checkpoints" / "BUSSAM_seed62.pth"
log_path = STUDY_ROOT / "logs" / "train_seed62.log"

if dest.exists():

    print("Seed 62 checkpoint already exists:")
    print(dest)

else:

    print("\n========================================")
    print("STARTING BUSSAM SEED 62")
    print("========================================\n")

    started = time.time()

    run_logged(
        [sys.executable, "train.py"],
        {
            "MICA_SEED": "62"
        },
        log_path
    )

    src = newest_checkpoint(started)

    shutil.copy2(src, dest)

    print("\n========================================")
    print("✅ SEED 62 COMPLETED")
    print("Checkpoint:", dest)
    print("========================================")

In [ ]:
import shutil
from pathlib import Path

project = Path("/kaggle/working/BUSSAM-main")

if project.exists():
    shutil.rmtree(project)
    print("✅ BUSSAM-main removed")
else:
    print("BUSSAM-main already removed")

In [ ]:
from pathlib import Path

study = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

for p in study.rglob("*"):
    if p.is_file():
        print(round(p.stat().st_size / (1024**2), 1), "MB", p)

In [ ]:
import shutil

source = "/kaggle/working/MICA2026_BUSSAM_STUDY"
zip_path = "/kaggle/working/MICA2026_BUSSAM_STUDY_backup"

shutil.make_archive(
    zip_path,
    "zip",
    source
)

print("✅ ZIP created:")
print(zip_path + ".zip")

## 6. BUSSAM prompt-robustness evaluation

Protocols: centre lesion point, deterministic random lesion point, boundary point, and 5%, 10%, and 20% displaced positive points. Results from normal/empty masks are retained but analysed separately. The cell temporarily changes only `load_path` in the configuration and restores it after every test.

In [ ]:
config_file=PROJECT_ROOT/'utils'/'config.py'
prompt_modes=['center','random_lesion','boundary','displaced_5','displaced_10','displaced_20']

def set_load_path_temporarily(checkpoint):
    original=config_file.read_text()
    changed=re.sub(r'(?m)^(\s*load_path\s*=\s*)["\'][^"\']+["\']',lambda m:m.group(1)+repr(str(checkpoint)),original)
    if changed==original: raise RuntimeError('Could not patch load_path in utils/config.py')
    config_file.write_text(changed); return original

def attach_manifest(case_csv,seed,prompt):
    c=pd.read_csv(case_csv); test=manifest[manifest.split=='test'].reset_index(drop=True)
    if len(c)!=len(test): raise ValueError(f'Per-case rows {len(c)} != test cases {len(test)}')
    c=pd.concat([test[['raw_id','stem','category','lesion_present','lesion_pixels']].reset_index(drop=True),c],axis=1)
    c['model']='BUSSAM'; c['seed']=seed; c['prompt']=prompt
    for col in ['dice','iou','accuracy','sensitivity','specificity']: c[col]=100*c[col]
    c.to_csv(case_csv,index=False)

if RUN_BUSSAM_ROBUSTNESS:
    for seed in SEEDS:
        ckpt=STUDY_ROOT/'checkpoints'/f'BUSSAM_seed{seed}.pth'
        if not ckpt.exists():
            supplied=PROJECT_ROOT/'outputs'/'BUSSAM_0803191842'/'checkpoints'/'BUSSAM_100.pth'
            if len(SEEDS)==1 and supplied.exists(): ckpt=supplied
            else: raise FileNotFoundError(f'Missing checkpoint {ckpt}')
        for mode in prompt_modes:
            case_csv = STUDY_ROOT / "per_case" / f"BUSSAM_seed{seed}_{mode}.csv"
            if case_csv.exists(): print('Result exists; skipped:',case_csv); continue
            original=set_load_path_temporarily(ckpt)
            try:
                run_logged([sys.executable,str(launcher)],
                    {'MICA_SEED':str(seed),'MICA_SCRIPT':'test.py','MICA_PROMPT_MODE':mode,
                     'MICA_PROMPT_SEED':str(seed),'MICA_CASE_LOG':str(case_csv)},
                    STUDY_ROOT/'logs'/f'test_seed{seed}_{mode}.log')
                attach_manifest(case_csv,seed,mode)
            finally: config_file.write_text(original)
else:
    print('Robustness evaluation disabled. Set RUN_BUSSAM_ROBUSTNESS=True when checkpoints are ready.')


In [ ]:
seed = 42
mode = "center"

ckpt = STUDY_ROOT / "checkpoints" / f"BUSSAM_seed{seed}.pth"
case_csv = STUDY_ROOT / "per_case" / f"BUSSAM_seed{seed}_{mode}.csv"
log_path = STUDY_ROOT / "logs" / f"test_seed{seed}_{mode}.log"

print("Checkpoint:", ckpt)
print("Exists:", ckpt.exists())

original = set_load_path_temporarily(ckpt)

try:
    run_logged(
        [sys.executable, str(launcher)],
        {
            "MICA_SEED": str(seed),
            "MICA_SCRIPT": "test.py",
            "MICA_PROMPT_MODE": mode,
            "MICA_PROMPT_SEED": str(seed),
            "MICA_CASE_LOG": str(case_csv),
        },
        log_path
    )
finally:
    config_file.write_text(original)

print("\n✅ Seed 42 + center evaluation completed")
print("CSV:", case_csv)
print("CSV exists:", case_csv.exists())

In [ ]:
if case_csv.exists():
    df = pd.read_csv(case_csv)
    print("Rows:", len(df))
    print(df.head())

In [ ]:
# Run all remaining BUSSAM robustness evaluations

prompt_modes = [
    "center",
    "random_lesion",
    "boundary",
    "displaced_5",
    "displaced_10",
    "displaced_20"
]

(STUDY_ROOT / "per_case").mkdir(parents=True, exist_ok=True)
(STUDY_ROOT / "logs").mkdir(parents=True, exist_ok=True)

for seed in SEEDS:

    ckpt = STUDY_ROOT / "checkpoints" / f"BUSSAM_seed{seed}.pth"

    if not ckpt.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt}")

    print(f"\n========== SEED {seed} ==========")

    for mode in prompt_modes:

        case_csv = (
            STUDY_ROOT
            / "per_case"
            / f"BUSSAM_seed{seed}_{mode}.csv"
        )

        # Skip already completed evaluation
        if case_csv.exists():
            print("✅ Already exists, skipped:", case_csv.name)
            continue

        print(f"\n▶ Running Seed {seed} | Prompt: {mode}")

        original = set_load_path_temporarily(ckpt)

        try:
            run_logged(
                [sys.executable, str(launcher)],
                {
                    "MICA_SEED": str(seed),
                    "MICA_SCRIPT": "test.py",
                    "MICA_PROMPT_MODE": mode,
                    "MICA_PROMPT_SEED": str(seed),
                    "MICA_CASE_LOG": str(case_csv),
                },
                STUDY_ROOT / "logs" / f"test_seed{seed}_{mode}.log"
            )

            # Add manifest information after successful evaluation
            attach_manifest(case_csv, seed, mode)

            print(
                f"✅ COMPLETED: Seed {seed} | {mode} | "
                f"{case_csv.name}"
            )

        finally:
            config_file.write_text(original)

print("\n🎉 ALL AVAILABLE ROBUSTNESS EVALUATIONS COMPLETED")

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    STUDY_ROOT / "split_manifest_audited.csv",
    STUDY_ROOT / "split_manifest_clean.csv",
    PROJECT_ROOT / "split_manifest_audited.csv",
    PROJECT_ROOT / "split_manifest_clean.csv",
]

for p in candidates:
    if p.exists():
        print("Found:", p)
        tmp = pd.read_csv(p)
        print(tmp.columns.tolist())
        print(tmp["split"].value_counts(dropna=False))

In [ ]:
case_csv = STUDY_ROOT / "per_case" / "BUSSAM_seed42_random_lesion.csv"

print("CSV exists:", case_csv.exists())
print("Rows:", len(pd.read_csv(case_csv)))

attach_manifest(case_csv, 42, "random_lesion")

print("✅ Seed 42 random_lesion metadata attached")

In [ ]:
import os
import torch

print("Main process CUDA:", torch.cuda.is_available())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("CUDA device count:", torch.cuda.device_count())

In [ ]:
!pip install thop

In [ ]:
from pathlib import Path
import os
import sys
import re
import time
import shutil
import subprocess
import pandas as pd

PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

os.chdir(PROJECT_ROOT)

# Load the audited clean manifest
manifest = pd.read_csv(
    STUDY_ROOT / "split_manifest_clean.csv"
)

SEEDS = [42, 52, 62]

RUN_BUSSAM_ROBUSTNESS = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STUDY_ROOT:", STUDY_ROOT)
print("Current directory:", Path.cwd())

# Verify all 3 checkpoints
for seed in SEEDS:
    ckpt = STUDY_ROOT / "checkpoints" / f"BUSSAM_seed{seed}.pth"
    print(f"Seed {seed} checkpoint:", ckpt.exists(),
          f"{ckpt.stat().st_size/(1024**2):.1f} MB" if ckpt.exists() else "")

# Recreate launcher
launcher = STUDY_ROOT / "seeded_launcher.py"

launcher.write_text(r'''
import os, sys, runpy, random
sys.path.insert(0, os.getcwd())

import numpy as np
import torch

seed = int(os.environ["MICA_SEED"])

os.environ["PYTHONHASHSEED"] = str(seed)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.argv = [os.environ["MICA_SCRIPT"]]

runpy.run_path(
    os.environ["MICA_SCRIPT"],
    run_name="__main__"
)
''')

# Recreate logging function
def run_logged(cmd, env, log_path):
    with open(log_path, "w") as log:
        proc = subprocess.Popen(
            cmd,
            cwd=PROJECT_ROOT,
            env={**os.environ, **env},
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()

        rc = proc.wait()

        if rc != 0:
            raise RuntimeError(
                f"Command failed ({rc}); see {log_path}"
            )

print("\n✅ Evaluation environment restored.")

## 7. Matched U-Net baseline

This baseline uses the same audited manifests, 256×256 inputs, BCE + Dice loss, three seeds, and lesion/empty-mask-aware per-case reporting. It is automatic and receives no point prompt; this distinction must be stated in the paper.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

class BUSIDataset(Dataset):
    def __init__(self,frame,augment=False): self.df=frame.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; x=cv2.imread(r.image,0); y=cv2.imread(r['mask'],0)
        x=cv2.resize(x,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_LINEAR).astype('float32')/255
        y=(cv2.resize(y,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_NEAREST)>0).astype('float32')
        if self.augment and random.random()<.5: x=np.fliplr(x).copy(); y=np.fliplr(y).copy()
        return torch.from_numpy(x[None]),torch.from_numpy(y[None]),r.raw_id

def block(a,b): return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
class UNet(nn.Module):
    def __init__(self):
        super().__init__(); self.e1=block(1,32); self.e2=block(32,64); self.e3=block(64,128); self.b=block(128,256)
        self.pool=nn.MaxPool2d(2); self.u3=nn.ConvTranspose2d(256,128,2,2); self.d3=block(256,128)
        self.u2=nn.ConvTranspose2d(128,64,2,2); self.d2=block(128,64); self.u1=nn.ConvTranspose2d(64,32,2,2); self.d1=block(64,32); self.out=nn.Conv2d(32,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b)); z=self.b(self.pool(c))
        z=self.d3(torch.cat([self.u3(z),c],1)); z=self.d2(torch.cat([self.u2(z),b],1)); z=self.d1(torch.cat([self.u1(z),a],1)); return self.out(z)

def dice_loss(logit,y,eps=1e-6):
    p=torch.sigmoid(logit); inter=(p*y).sum((1,2,3)); den=p.sum((1,2,3))+y.sum((1,2,3)); return (1-(2*inter+eps)/(den+eps)).mean()

def case_metrics(pred,gt):
    pred=pred.astype(bool); gt=gt.astype(bool); tp=(pred & gt).sum(); fp=(pred & ~gt).sum(); fn=(~pred & gt).sum(); tn=(~pred & ~gt).sum(); eps=1e-8
    both_empty=not pred.any() and not gt.any()
    dice=1.0 if both_empty else (2*tp+eps)/(2*tp+fp+fn+eps); iou=1.0 if both_empty else (tp+eps)/(tp+fp+fn+eps)
    se=1.0 if not gt.any() and not pred.any() else (tp+eps)/(tp+fn+eps); sp=(tn+eps)/(tn+fp+eps); acc=(tp+tn)/(tp+tn+fp+fn)
    from scipy.ndimage import binary_erosion
    from scipy.spatial.distance import directed_hausdorff
    if both_empty: hd=0.0
    elif not pred.any() or not gt.any(): hd=float(np.hypot(*pred.shape))
    else:
        pe=np.argwhere(pred^binary_erosion(pred)); ge=np.argwhere(gt^binary_erosion(gt)); hd=max(directed_hausdorff(pe,ge)[0],directed_hausdorff(ge,pe)[0])
    return dice*100,iou*100,acc*100,se*100,sp*100,hd

def run_unet(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr=DataLoader(BUSIDataset(manifest[manifest.split=='train'],True),8,shuffle=True,num_workers=2,pin_memory=True)
    va=DataLoader(BUSIDataset(manifest[manifest.split=='val']),8,shuffle=False,num_workers=2)
    te=DataLoader(BUSIDataset(manifest[manifest.split=='test']),1,shuffle=False,num_workers=2)
    model=UNet().cuda(); opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4); bce=nn.BCEWithLogitsLoss(); best=-1
    ck=STUDY_ROOT/'checkpoints'/f'UNet_seed{seed}.pth'; history=[]
    for epoch in range(1,UNET_EPOCHS+1):
        model.train(); losses=[]
        for x,y,_ in tr:
            x=x.cuda(non_blocking=True); y=y.cuda(non_blocking=True); opt.zero_grad(); z=model(x); loss=.2*bce(z,y)+.8*dice_loss(z,y); loss.backward(); opt.step(); losses.append(loss.item())
        model.eval(); vd=[]
        with torch.no_grad():
            for x,y,_ in va:
                p=(torch.sigmoid(model(x.cuda()))>.5).cpu().numpy(); g=y.numpy()
                vd += [case_metrics(p[i,0],g[i,0])[0] for i in range(len(x))]
        score=float(np.mean(vd)); history.append({'epoch':epoch,'train_loss':np.mean(losses),'val_dice':score})
        if score>best: best=score; torch.save(model.state_dict(),ck)
        if epoch==1 or epoch%10==0: print(seed,epoch,round(np.mean(losses),4),round(score,3))
    pd.DataFrame(history).to_csv(STUDY_ROOT/'logs'/f'UNet_seed{seed}_history.csv',index=False)
    model.load_state_dict(torch.load(ck,map_location='cuda')); model.eval(); out=[]
    testmeta=manifest[manifest.split=='test'].reset_index(drop=True)
    with torch.no_grad():
        for i,(x,y,raw) in enumerate(te):
            p=(torch.sigmoid(model(x.cuda()))>.5).cpu().numpy()[0,0]; g=y.numpy()[0,0]
            d,j,a,se,sp,hd=case_metrics(p,g); r=testmeta.iloc[i]
            out.append({'raw_id':r.raw_id,'stem':r.stem,'category':r.category,'lesion_present':r.lesion_present,'lesion_pixels':r.lesion_pixels,
                        'model':'UNet','seed':seed,'prompt':'none','dice':d,'iou':j,'accuracy':a,'sensitivity':se,'specificity':sp,'hausdorff':hd})
    pd.DataFrame(out).to_csv(STUDY_ROOT/'per_case'/f'UNet_seed{seed}.csv',index=False)

if RUN_UNET_BASELINE:
    for seed in SEEDS:
        out=STUDY_ROOT/'per_case'/f'UNet_seed{seed}.csv'
        if out.exists(): print('Baseline exists; skipped:',out)
        else: run_unet(seed)
else: print('U-Net baseline disabled. Set RUN_UNET_BASELINE=True when ready.')


In [ ]:
from pathlib import Path

ckpt_dir = STUDY_ROOT / "checkpoints"

for seed in SEEDS:
    p = ckpt_dir / f"UNet_seed{seed}.pth"
    print(f"Seed {seed}: {p.exists()} | {p.stat().st_size / (1024**2):.2f} MB" if p.exists()
          else f"Seed {seed}: ❌ missing")

In [ ]:
for seed in SEEDS:
    p = STUDY_ROOT / "per_case" / f"UNet_seed{seed}.csv"
    print(f"Seed {seed}: {p.exists()} | {len(pd.read_csv(p)) if p.exists() else 0} rows")

In [ ]:
import pandas as pd
from pathlib import Path

for seed in SEEDS:
    p = STUDY_ROOT / "per_case" / f"UNet_seed{seed}.csv"
    df = pd.read_csv(p)

    print(f"\n===== U-Net Seed {seed} =====")
    for col in ["dice", "iou", "accuracy", "sensitivity", "specificity", "hausdorff"]:
        print(f"{col:12s}: mean={df[col].mean():.4f}, std={df[col].std():.4f}")

In [ ]:
from pathlib import Path
import pandas as pd

STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")
PER_CASE = STUDY_ROOT / "per_case"

print("STUDY_ROOT exists:", STUDY_ROOT.exists())
print("per_case exists:", PER_CASE.exists())

if PER_CASE.exists():
    files = sorted(PER_CASE.glob("*.csv"))
    print("Per-case CSV count:", len(files))
    for f in files:
        df = pd.read_csv(f)
        print(f.name, "| rows =", len(df))

In [ ]:
# ============================================================
# REGENERATE BUSSAM 18 PER-CASE CSVs
# 3 seeds × 6 prompts = 18 evaluations
# Uses existing BUSSAM checkpoints — NO TRAINING
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import shutil
import re

PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

TEST_PY = PROJECT_ROOT / "test.py"
CONFIG_PY = PROJECT_ROOT / "utils" / "config.py"

CHECKPOINT_DIR = STUDY_ROOT / "checkpoints"
PER_CASE_DIR = STUDY_ROOT / "per_case"
LOG_DIR = STUDY_ROOT / "logs"

PER_CASE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 52, 62]

PROMPT_MODES = [
    "center",
    "random_lesion",
    "boundary",
    "displaced_5",
    "displaced_10",
    "displaced_20",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("test.py exists:", TEST_PY.exists())
print("config.py exists:", CONFIG_PY.exists())

print("\nCheckpoints:")
for seed in SEEDS:
    ck = CHECKPOINT_DIR / f"BUSSAM_seed{seed}.pth"
    print(seed, "->", ck.exists(), ck)

# ------------------------------------------------------------
# Backup config.py
# ------------------------------------------------------------

backup_config = CONFIG_PY.with_suffix(".py.before_regeneration_backup")

if not backup_config.exists():
    shutil.copy2(CONFIG_PY, backup_config)
    print("\nConfig backup created:", backup_config)
else:
    print("\nConfig backup already exists:", backup_config)

original_config = CONFIG_PY.read_text()

# ------------------------------------------------------------
# Helper: patch BUSI load_path only
# ------------------------------------------------------------

def patch_busi_checkpoint(checkpoint_path):
    text = CONFIG_PY.read_text()

    # Find BUSI section and replace its load_path.
    marker = "data_subpath = 'datasets/BUSI/'"

    if marker not in text:
        raise RuntimeError("Could not find BUSI section in config.py")

    pos = text.index(marker)

    # Search only after BUSI marker
    load_pos = text.find("load_path =", pos)

    if load_pos == -1:
        raise RuntimeError("Could not find BUSI load_path")

    line_end = text.find("\n", load_pos)
    if line_end == -1:
        line_end = len(text)

    old_line = text[load_pos:line_end]

    new_line = f"load_path = '{checkpoint_path}'  # BUSI checkpoint"

    text = text[:load_pos] + new_line + text[line_end:]

    CONFIG_PY.write_text(text)

    print("Patched:")
    print(new_line)


# ------------------------------------------------------------
# Run one evaluation
# ------------------------------------------------------------

def run_one(seed, prompt):
    checkpoint = CHECKPOINT_DIR / f"BUSSAM_seed{seed}.pth"

    output_csv = PER_CASE_DIR / f"BUSSAM_seed{seed}_{prompt}.csv"
    log_file = LOG_DIR / f"test_seed{seed}_{prompt}.log"

    if not checkpoint.exists():
        raise FileNotFoundError(f"Checkpoint missing: {checkpoint}")

    # If already regenerated, don't waste GPU time.
    if output_csv.exists():
        print(f"SKIP existing: {output_csv.name}")
        return True

    print("\n" + "=" * 70)
    print(f"SEED {seed} | PROMPT {prompt}")
    print("Checkpoint:", checkpoint)
    print("=" * 70)

    # Patch checkpoint
    patch_busi_checkpoint(checkpoint)

    env = os.environ.copy()

    # Make project imports work
    env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

    # Prompt controls used by patched generate_prompts.py
    env["MICA_PROMPT_MODE"] = prompt
    env["MICA_PROMPT_SEED"] = str(seed)

    # Tell patched metrics.py where to write per-case output
    env["MICA_CASE_LOG"] = str(output_csv)

    # Useful for reproducibility
    env["PYTHONHASHSEED"] = str(seed)

    cmd = [
        sys.executable,
        str(TEST_PY),
        "--modelname", "BUSSAM",
        "--task", "BUSI",
        "--batch_size", "8",
        "--n_gpu", "1",
    ]

    print("Running:")
    print(" ".join(cmd))
    print("MICA_PROMPT_MODE =", prompt)
    print("MICA_PROMPT_SEED =", seed)
    print("MICA_CASE_LOG    =", output_csv)

    with open(log_file, "w") as lf:
        result = subprocess.run(
            cmd,
            cwd=str(PROJECT_ROOT),
            env=env,
            stdout=lf,
            stderr=subprocess.STDOUT,
            text=True,
        )

    print("Return code:", result.returncode)

    if result.returncode != 0:
        print("\n❌ FAILED")
        print("Log:", log_file)

        # Show last 40 lines of log
        lines = log_file.read_text(errors="replace").splitlines()
        print("\n----- LAST LOG LINES -----")
        for line in lines[-40:]:
            print(line)

        return False

    # Verify actual CSV
    if not output_csv.exists():
        print("\n❌ Test completed but CSV was NOT created")
        print("Expected:", output_csv)

        lines = log_file.read_text(errors="replace").splitlines()
        print("\n----- LAST LOG LINES -----")
        for line in lines[-30:]:
            print(line)

        return False

    # Basic validation
    import pandas as pd

    df = pd.read_csv(output_csv)

    print("\n✅ COMPLETED")
    print("CSV:", output_csv)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    if "dice" in df.columns:
        print("Dice mean:", round(df["dice"].mean(), 4))

    if len(df) != 121:
        print("⚠️ WARNING: Expected 121 rows, got", len(df))

    return True


# ------------------------------------------------------------
# Run all 18
# ------------------------------------------------------------

completed = []
failed = []

try:
    for seed in SEEDS:
        for prompt in PROMPT_MODES:

            ok = run_one(seed, prompt)

            if ok:
                completed.append((seed, prompt))
            else:
                failed.append((seed, prompt))

finally:
    # Restore original config.py
    CONFIG_PY.write_text(original_config)
    print("\nOriginal config.py restored.")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n\n" + "=" * 70)
print("FINAL BUSSAM REGENERATION STATUS")
print("=" * 70)

print("\nCompleted:", len(completed))
for seed, prompt in completed:
    print(f"  ✅ seed={seed:2d} | {prompt}")

print("\nFailed:", len(failed))
for seed, prompt in failed:
    print(f"  ❌ seed={seed:2d} | {prompt}")

print("\nPer-case CSVs currently present:")
csvs = sorted(PER_CASE_DIR.glob("BUSSAM_seed*.csv"))
print("Count:", len(csvs))

for f in csvs:
    try:
        import pandas as pd
        df = pd.read_csv(f)
        print(f"{f.name:42s} rows={len(df)}")
    except Exception:
        print(f.name)

In [ ]:
!pip install -q batchgenerators thop

In [ ]:
import batchgenerators
import thop

print("✅ batchgenerators OK")
print("✅ thop OK")

## 8. Aggregate results, bootstrap confidence intervals, and statistical comparisons

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

PER_CASE_DIR = STUDY_ROOT / "per_case"
TABLES_DIR = STUDY_ROOT / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAPS = 2000

PROMPT_MODES = [
    "center",
    "random_lesion",
    "boundary",
    "displaced_5",
    "displaced_10",
    "displaced_20"
]

print("✅ Analysis environment ready")
print("Per-case:", PER_CASE_DIR.exists())
print("Tables:", TABLES_DIR.exists())
print("BUSSAM CSVs:", len(list(PER_CASE_DIR.glob("BUSSAM_seed*.csv"))))
print("UNet CSVs:", len(list(PER_CASE_DIR.glob("UNet_seed*.csv"))))

In [ ]:
# ============================================================
# CELL 8 — FINAL STATISTICAL ANALYSIS
# BUSSAM robustness + U-Net baseline
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")
PER_CASE_DIR = STUDY_ROOT / "per_case"
TABLES_DIR = STUDY_ROOT / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

BOOTSTRAPS = 2000

PROMPT_MODES = [
    "center",
    "random_lesion",
    "boundary",
    "displaced_5",
    "displaced_10",
    "displaced_20"
]

# ------------------------------------------------------------
# 1. Load all per-case CSVs
# ------------------------------------------------------------

files = sorted(PER_CASE_DIR.glob("*.csv"))

print("Per-case files found:", len(files))

for f in files:
    print(" ", f.name)

if len(files) == 0:
    raise FileNotFoundError("No per-case CSV files found.")

results = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print("\nTotal rows:", len(results))
print("Models:", results["model"].dropna().unique()
      if "model" in results.columns else "model column missing")

# ------------------------------------------------------------
# 2. Fix BUSSAM CSV format
# ------------------------------------------------------------

# Some old BUSSAM CSVs may have different formats.
# Add missing metadata from filename where possible.

for col in [
    "raw_id",
    "stem",
    "category",
    "lesion_present",
    "lesion_pixels",
    "model",
    "seed",
    "prompt"
]:
    if col not in results.columns:
        results[col] = np.nan

# Convert numeric columns
metric_cols = [
    "dice",
    "iou",
    "accuracy",
    "sensitivity",
    "specificity",
    "hausdorff"
]

for col in metric_cols:
    if col in results.columns:
        results[col] = pd.to_numeric(
            results[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# 3. Normalize metric scale
# ------------------------------------------------------------
# U-Net results are already percentages.
# BUSSAM center may be stored as 0-1 in regenerated CSVs.
# Convert BUSSAM metrics to percentage when needed.

for col in [
    "dice",
    "iou",
    "accuracy",
    "sensitivity",
    "specificity"
]:
    if col in results.columns:

        mask = (
            (results["model"] == "BUSSAM") &
            results[col].notna() &
            (results[col] <= 1.000001)
        )

        results.loc[mask, col] *= 100

# ------------------------------------------------------------
# 4. Save combined results
# ------------------------------------------------------------

all_results_path = STUDY_ROOT / "all_per_case_results.csv"

results.to_csv(
    all_results_path,
    index=False
)

print("\n✅ Combined results saved:")
print(all_results_path)

print("\nRows:", len(results))

# ------------------------------------------------------------
# 5. Bootstrap CI
# ------------------------------------------------------------

def bootstrap_ci(values, n=2000, seed=2026):

    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]

    if len(v) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)

    samples = rng.choice(
        v,
        size=(n, len(v)),
        replace=True
    )

    means = samples.mean(axis=1)

    return (
        float(np.mean(v)),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975))
    )

# ------------------------------------------------------------
# 6. Bootstrap summary
# ------------------------------------------------------------

summary_rows = []

group_cols = [
    "model",
    "prompt",
    "seed",
    "lesion_present"
]

for keys, g in results.groupby(
    group_cols,
    dropna=False
):

    for metric in metric_cols:

        mean, lo, hi = bootstrap_ci(
            g[metric].values,
            n=BOOTSTRAPS
        )

        row = dict(
            zip(group_cols, keys)
        )

        row.update({
            "metric": metric,
            "n": len(g),
            "mean": mean,
            "ci_low": lo,
            "ci_high": hi
        })

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

summary_path = TABLES_DIR / "bootstrap_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

print("\n✅ Bootstrap summary saved:")
print(summary_path)

# Show BUSSAM lesion-present Dice
print("\n===== BUSSAM DICE =====")

display(
    summary[
        (summary["model"] == "BUSSAM") &
        (summary["metric"] == "dice") &
        (summary["lesion_present"] == True)
    ].round(3)
)

# ------------------------------------------------------------
# 7. Prompt-wise paired comparisons
# ------------------------------------------------------------

print("\n===== PAIRED PROMPT TESTS =====")

stats_rows = []

b = results[
    (results["model"] == "BUSSAM") &
    (results["lesion_present"] == True)
].copy()

for seed in sorted(
    b["seed"].dropna().unique()
):

    base = b[
        (b["seed"] == seed) &
        (b["prompt"] == "center")
    ][
        ["raw_id", "dice"]
    ].rename(
        columns={"dice": "base"}
    )

    for prompt in PROMPT_MODES:

        if prompt == "center":
            continue

        alt = b[
            (b["seed"] == seed) &
            (b["prompt"] == prompt)
        ][
            ["raw_id", "dice"]
        ].rename(
            columns={"dice": "alt"}
        )

        # If raw_id unavailable, use case_index
        if len(base) == 0 or len(alt) == 0:
            continue

        m = base.merge(
            alt,
            on="raw_id",
            how="inner"
        )

        if len(m) == 0:
            continue

        delta = m["alt"] - m["base"]

        # Identical arrays need special handling
        if np.allclose(
            m["base"].values,
            m["alt"].values
        ):
            stat = 0.0
            p = 1.0
        else:
            try:
                stat, p = wilcoxon(
                    m["base"],
                    m["alt"],
                    zero_method="zsplit"
                )
            except ValueError:
                stat = 0.0
                p = 1.0

        stats_rows.append({
            "seed": int(seed),
            "comparison": f"center vs {prompt}",
            "n": len(m),
            "median_delta": float(np.median(delta)),
            "mean_delta": float(np.mean(delta)),
            "p_raw": float(p)
        })

stats = pd.DataFrame(stats_rows)

# ------------------------------------------------------------
# 8. Holm correction
# ------------------------------------------------------------

if len(stats):

    stats = stats.sort_values(
        "p_raw"
    ).reset_index(drop=True)

    m_tests = len(stats)

    stats["p_holm"] = np.nan

    for i in range(m_tests):

        stats.loc[i, "p_holm"] = min(
            1.0,
            stats.loc[i, "p_raw"] *
            (m_tests - i)
        )

    # enforce monotonicity
    stats["p_holm"] = np.maximum.accumulate(
        stats["p_holm"].values
    )

stats_path = TABLES_DIR / "paired_prompt_tests_holm.csv"

stats.to_csv(
    stats_path,
    index=False
)

print("\n✅ Prompt statistical tests saved:")
print(stats_path)

if len(stats):
    display(stats.round(4))
else:
    print("⚠️ No paired prompt comparisons available.")

# ------------------------------------------------------------
# 9. Prompt-level aggregate
# ------------------------------------------------------------

print("\n===== PROMPT-LEVEL BUSSAM RESULTS =====")

prompt_summary = (
    b.groupby("prompt")[
        metric_cols
    ]
    .agg(["mean", "std"])
)

display(prompt_summary.round(3))

prompt_summary_path = (
    TABLES_DIR /
    "BUSSAM_prompt_summary.csv"
)

prompt_summary.to_csv(
    prompt_summary_path
)

print(
    "\nSaved:",
    prompt_summary_path
)

# ------------------------------------------------------------
# 10. Seed-level aggregate
# ------------------------------------------------------------

print("\n===== SEED-LEVEL BUSSAM RESULTS =====")

seed_summary = (
    b.groupby("seed")[
        metric_cols
    ]
    .agg(["mean", "std"])
)

display(seed_summary.round(3))

seed_summary_path = (
    TABLES_DIR /
    "BUSSAM_seed_summary.csv"
)

seed_summary.to_csv(
    seed_summary_path
)

print(
    "\nSaved:",
    seed_summary_path
)

# ------------------------------------------------------------
# 11. U-Net baseline
# ------------------------------------------------------------

print("\n===== U-NET BASELINE =====")

unet = results[
    results["model"] == "UNet"
].copy()

if len(unet):

    unet_summary = (
        unet[metric_cols]
        .agg(["mean", "std"])
    )

    display(
        unet_summary.round(3)
    )

    unet_path = (
        TABLES_DIR /
        "UNet_summary.csv"
    )

    unet_summary.to_csv(
        unet_path
    )

    print(
        "Saved:",
        unet_path
    )

else:
    print("⚠️ No U-Net rows found.")

# ------------------------------------------------------------
# 12. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 8 COMPLETE")
print("=" * 70)

print("Total CSVs:", len(files))
print("Total rows:", len(results))
print("BUSSAM rows:", len(results[results["model"] == "BUSSAM"]))
print("U-Net rows:", len(results[results["model"] == "UNet"]))

print("\nGenerated tables:")

for f in sorted(TABLES_DIR.glob("*.csv")):
    print(" ", f.name)

print("\n✅ No GPU required for the next analysis steps.")

NOVALITY START

In [ ]:
# ============================================================
# NOVELTY MODULE 1
# Entropy-Adaptive Soft-ROI Enhancement
# + Paired Stability Analysis
# + Boundary Uncertainty
# + Reliability Score
# ============================================================

import cv2
import numpy as np
import torch
import pandas as pd
from pathlib import Path


def normalize_uint8(img):
    img = img.astype(np.float32)
    mn, mx = img.min(), img.max()

    if mx - mn < 1e-8:
        return np.zeros_like(img, dtype=np.uint8)

    img = (img - mn) / (mx - mn)
    return (img * 255).astype(np.uint8)


def image_entropy(img):
    """
    Calculate normalized entropy of ultrasound image.
    Higher entropy = more complex/noisy region.
    """
    img = normalize_uint8(img)

    hist = cv2.calcHist(
        [img],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    hist = hist / (hist.sum() + 1e-8)

    entropy = -np.sum(
        hist * np.log2(hist + 1e-12)
    )

    return entropy / 8.0


def entropy_adaptive_clahe(img):
    """
    Adaptive CLAHE strength based on image entropy.
    """

    img = normalize_uint8(img)

    ent = image_entropy(img)

    # Low entropy -> stronger enhancement
    # High entropy -> weaker enhancement
    clip_limit = 3.0 - 1.5 * ent

    clip_limit = float(
        np.clip(clip_limit, 1.0, 3.0)
    )

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(img)

    return enhanced, ent


def create_soft_roi(mask, dilation_ratio=0.15):
    """
    Create dilated soft ROI around predicted lesion.
    """

    mask = (mask > 0).astype(np.uint8)

    ys, xs = np.where(mask > 0)

    if len(xs) == 0:
        return np.ones_like(mask, dtype=np.uint8)

    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()

    w = x2 - x1 + 1
    h = y2 - y1 + 1

    dx = int(w * dilation_ratio)
    dy = int(h * dilation_ratio)

    x1 = max(0, x1 - dx)
    x2 = min(mask.shape[1] - 1, x2 + dx)

    y1 = max(0, y1 - dy)
    y2 = min(mask.shape[0] - 1, y2 + dy)

    roi = np.zeros_like(mask)

    roi[y1:y2+1, x1:x2+1] = 1

    return roi


def edge_preserving_filter(img):
    """
    Preserve lesion boundaries while reducing ultrasound noise.
    """

    img = normalize_uint8(img)

    filtered = cv2.bilateralFilter(
        img,
        d=7,
        sigmaColor=40,
        sigmaSpace=40
    )

    return filtered


def enhance_with_soft_roi(image, predicted_mask):
    """
    Main proposed enhancement pipeline.

    Enhancement is restricted to the lesion soft ROI.
    Background remains unchanged.
    """

    image = normalize_uint8(image)

    roi = create_soft_roi(predicted_mask)

    enhanced, entropy = entropy_adaptive_clahe(image)

    enhanced = edge_preserving_filter(enhanced)

    output = image.copy()

    roi_bool = roi.astype(bool)

    output[roi_bool] = enhanced[roi_bool]

    return output, roi, entropy


# ============================================================
# PAIRED STABILITY
# ============================================================

def dice_score(mask1, mask2):
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    intersection = np.logical_and(
        mask1,
        mask2
    ).sum()

    return (
        2.0 * intersection
        /
        (mask1.sum() + mask2.sum() + 1e-8)
    )


def iou_score(mask1, mask2):
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    intersection = np.logical_and(
        mask1,
        mask2
    ).sum()

    union = np.logical_or(
        mask1,
        mask2
    ).sum()

    return intersection / (union + 1e-8)


def mask_displacement(mask1, mask2):
    """
    Measures centroid displacement between
    original and enhanced predictions.
    """

    def centroid(mask):
        ys, xs = np.where(mask > 0)

        if len(xs) == 0:
            return np.array([0.0, 0.0])

        return np.array([
            xs.mean(),
            ys.mean()
        ])

    c1 = centroid(mask1)
    c2 = centroid(mask2)

    return float(
        np.linalg.norm(c1 - c2)
    )


def boundary_uncertainty(mask1, mask2):
    """
    Uncertainty estimated from disagreement
    between original and enhanced masks.
    """

    mask1 = mask1.astype(np.uint8)
    mask2 = mask2.astype(np.uint8)

    disagreement = np.logical_xor(
        mask1,
        mask2
    )

    total = np.logical_or(
        mask1,
        mask2
    ).sum()

    if total == 0:
        return 1.0

    return float(
        disagreement.sum() /
        (total + 1e-8)
    )


def stability_score(mask1, mask2):
    """
    Combined paired stability score.
    """

    dice = dice_score(mask1, mask2)
    iou = iou_score(mask1, mask2)

    displacement = mask_displacement(
        mask1,
        mask2
    )

    # Normalize displacement
    displacement_factor = np.exp(
        -displacement / 20.0
    )

    score = (
        0.50 * dice +
        0.30 * iou +
        0.20 * displacement_factor
    )

    return float(
        np.clip(score, 0.0, 1.0)
    )


def reliability_score(
    stability,
    uncertainty,
    prediction_confidence
):
    """
    Final reliability score.

    Higher = more trustworthy prediction.
    """

    score = (
        0.50 * stability +
        0.25 * (1.0 - uncertainty) +
        0.25 * prediction_confidence
    )

    return float(
        np.clip(score, 0.0, 1.0)
    )


def selective_decision(
    reliability,
    threshold=0.70
):
    """
    Reliability-based abstention.

    Reliable -> ACCEPT
    Unreliable -> ABSTAIN
    """

    if reliability >= threshold:
        return "ACCEPT"

    return "ABSTAIN"


print("Novel BUSSAM reliability module loaded successfully.")

In [ ]:
# Check what variables are available after BUSSAM prediction

print("Available prediction-related variables:")

for name in dir():
    if any(x in name.lower() for x in [
        "mask", "pred", "output", "result"
    ]):
        print(name)

In [ ]:
print("mask1 :", type(mask1), np.shape(mask1))
print("mask2 :", type(mask2), np.shape(mask2))

print("mask1 foreground:", np.sum(np.asarray(mask1) > 0))
print("mask2 foreground:", np.sum(np.asarray(mask2) > 0))

In [ ]:
# ============================================================
# NOVELTY RESULT: PAIRED STABILITY + UNCERTAINTY
# ============================================================

original_mask = (mask1 > 0).astype(np.uint8)
enhanced_mask = (mask2 > 0).astype(np.uint8)

stability = stability_score(
    original_mask,
    enhanced_mask
)

uncertainty = boundary_uncertainty(
    original_mask,
    enhanced_mask
)

displacement = mask_displacement(
    original_mask,
    enhanced_mask
)

dice_consistency = dice_score(
    original_mask,
    enhanced_mask
)

iou_consistency = iou_score(
    original_mask,
    enhanced_mask
)

print("=" * 60)
print("NOVEL BUSSAM STABILITY ANALYSIS")
print("=" * 60)

print(f"Original foreground   : {original_mask.sum()}")
print(f"Enhanced foreground   : {enhanced_mask.sum()}")
print(f"Dice consistency      : {dice_consistency:.4f}")
print(f"IoU consistency       : {iou_consistency:.4f}")
print(f"Boundary uncertainty  : {uncertainty:.4f}")
print(f"Mask displacement     : {displacement:.3f}")
print(f"Stability score       : {stability:.4f}")

print("=" * 60)

In [ ]:
result = {
    "stability": stability_score(
        original_mask,
        enhanced_mask
    ),

    "uncertainty": boundary_uncertainty(
        original_mask,
        enhanced_mask
    ),

    "displacement": mask_displacement(
        original_mask,
        enhanced_mask
    ),

    "dice_consistency": dice_score(
        original_mask,
        enhanced_mask
    ),

    "iou_consistency": iou_score(
        original_mask,
        enhanced_mask
    )
}

result["reliability"] = reliability_score(
    result["stability"],
    result["uncertainty"],
    prediction_confidence=0.90
)

result["decision"] = selective_decision(
    result["reliability"]
)

print(pd.DataFrame([result]))

In [ ]:
# ============================================================
# NOVELTY MODULE 2
# Reliability + Selective Prediction
# ============================================================

prediction_confidence = 0.90

reliability = reliability_score(
    stability=stability,
    uncertainty=uncertainty,
    prediction_confidence=prediction_confidence
)

decision = selective_decision(
    reliability,
    threshold=0.70
)

print("=" * 60)
print("SELECTIVE RELIABILITY RESULT")
print("=" * 60)

print(f"Prediction confidence : {prediction_confidence:.3f}")
print(f"Stability score       : {stability:.3f}")
print(f"Boundary uncertainty  : {uncertainty:.3f}")
print(f"Reliability score     : {reliability:.3f}")
print(f"Decision               : {decision}")

if decision == "ACCEPT":
    print("\n✓ Prediction accepted as reliable.")
else:
    print("\n⚠ Prediction rejected/abstained.")
    print("→ Radiologist review required.")

print("=" * 60)

In [ ]:
# ============================================================
# NOVELTY MODULE 3
# CURRENT CASE RESULT TABLE
# ============================================================

novel_result = pd.DataFrame([{
    "original_foreground": int(original_mask.sum()),
    "enhanced_foreground": int(enhanced_mask.sum()),
    "dice_consistency": dice_consistency,
    "iou_consistency": iou_consistency,
    "boundary_uncertainty": uncertainty,
    "mask_displacement": displacement,
    "stability_score": stability,
    "prediction_confidence": prediction_confidence,
    "reliability_score": reliability,
    "decision": decision
}])

display(novel_result)

In [ ]:
print("all_results type:", type(all_results))

if isinstance(all_results, pd.DataFrame):
    print("Shape:", all_results.shape)
    print("Columns:")
    print(all_results.columns.tolist())
    display(all_results.head())

elif isinstance(all_results, list):
    print("Number of results:", len(all_results))
    print("First result:")
    print(all_results[0] if len(all_results) > 0 else "EMPTY")

else:
    print(all_results)

In [ ]:
# ============================================================
# NOVELTY EVALUATION - 10 EXISTING CASES
# ============================================================

df = pd.DataFrame(all_results)

print("Number of cases:", len(df))
print("\nAvailable metrics:")
print(df.columns.tolist())

print("\n===== NOVELTY RESULTS =====")

summary_cols = [
    "raw_id",
    "stability",
    "mean_uncertainty",
    "reliability",
    "center_prompt_stability",
    "selected_prompt_stability",
    "center_uncertainty",
    "selected_prompt_uncertainty"
]

display(df[summary_cols])

print("\n===== AVERAGE =====")

numeric_cols = [
    "stability",
    "mean_uncertainty",
    "reliability",
    "center_prompt_stability",
    "selected_prompt_stability",
    "center_uncertainty",
    "selected_prompt_uncertainty"
]

print(df[numeric_cols].mean())

In [ ]:
# ============================================================
# NOVELTY MODULE 4
# RISK-COVERAGE ANALYSIS
# ============================================================

df = pd.DataFrame(all_results).copy()

thresholds = np.arange(0.40, 0.96, 0.05)

coverage_results = []

for threshold in thresholds:

    accepted = df["reliability"] >= threshold

    coverage = accepted.mean()

    if accepted.sum() > 0:
        accepted_uncertainty = df.loc[
            accepted,
            "mean_uncertainty"
        ].mean()

        risk = accepted_uncertainty
    else:
        risk = np.nan

    coverage_results.append({
        "threshold": round(threshold, 2),
        "coverage": coverage,
        "risk": risk,
        "accepted_cases": int(accepted.sum()),
        "abstained_cases": int((~accepted).sum())
    })

risk_coverage_df = pd.DataFrame(
    coverage_results
)

print("=" * 70)
print("RISK-COVERAGE ANALYSIS")
print("=" * 70)

display(risk_coverage_df)

print("\nBest reliability threshold based on lowest risk:")
display(
    risk_coverage_df
    .sort_values("risk", na_position="last")
    .head(1)
)

In [ ]:
# ============================================================
# RELIABILITY THRESHOLD SUMMARY
# ============================================================

threshold = 0.70

df = pd.DataFrame(all_results).copy()

df["decision"] = np.where(
    df["reliability"] >= threshold,
    "ACCEPT",
    "ABSTAIN"
)

print("=" * 70)
print("SELECTIVE PREDICTION @ RELIABILITY THRESHOLD =", threshold)
print("=" * 70)

print("\nTotal cases    :", len(df))
print("Accepted cases :", (df["decision"] == "ACCEPT").sum())
print("Abstained cases:", (df["decision"] == "ABSTAIN").sum())

print(
    "Coverage       :",
    round((df["decision"] == "ACCEPT").mean(), 4)
)

print(
    "Abstention rate:",
    round((df["decision"] == "ABSTAIN").mean(), 4)
)

print("\nCase-wise decision:")
display(
    df[
        [
            "raw_id",
            "reliability",
            "stability",
            "mean_uncertainty",
            "decision"
        ]
    ]
)

In [ ]:
# ============================================================
# CHECK AVAILABLE GROUND-TRUTH / EVALUATION DATA
# ============================================================

print("Available GT / evaluation variables:\n")

for name in dir():
    name_lower = name.lower()

    if any(x in name_lower for x in [
        "gt", "ground", "target", "label",
        "truth", "dice", "iou", "hd95"
    ]):
        try:
            obj = eval(name)
            print(
                f"{name:30s} | "
                f"type={type(obj).__name__} | "
                f"shape={getattr(obj, 'shape', 'N/A')}"
            )
        except:
            pass

In [ ]:
# ============================================================
# ACTUAL GT VALIDATION
# Reliability vs Segmentation Quality
# ============================================================

gt_mask = (gt > 0).astype(np.uint8)

actual_dice = dice_score(
    enhanced_mask,
    gt_mask
)

actual_iou = iou_score(
    enhanced_mask,
    gt_mask
)

print("=" * 60)
print("ACTUAL GROUND-TRUTH VALIDATION")
print("=" * 60)

print(f"Ground-truth foreground : {gt_mask.sum()}")
print(f"Predicted foreground    : {enhanced_mask.sum()}")
print(f"Actual Dice             : {actual_dice:.4f}")
print(f"Actual IoU              : {actual_iou:.4f}")
print(f"Reliability score       : {reliability:.4f}")
print(f"Decision                : {decision}")
print("=" * 60)

In [ ]:
print("GT shape:", gt.shape)
print("GT unique values:", np.unique(gt))
print("GT non-zero pixels:", np.count_nonzero(gt))

print("\nAvailable mask-related paths:")
for name in ["mask_path", "mask_np", "masks", "predictions", "results"]:
    if name in globals():
        obj = globals()[name]
        print(name, type(obj), getattr(obj, "shape", ""))

In [ ]:
print("Current mask_path:", mask_path)
print("Current mask_np foreground:", np.count_nonzero(mask_np))

print("\nMASKS DICT:")
print(type(masks))
print("keys:", list(masks.keys())[:10])

print("\nPREDICTIONS DICT:")
print(type(predictions))
print("keys:", list(predictions.keys())[:10])

In [ ]:
# ============================================================
# FIND CORRECT GROUND-TRUTH MASK FOR CURRENT CASE
# ============================================================

from pathlib import Path

case_id = "benign_benign_104"

label_root = Path("/kaggle/working/BUSSAM-main/datasets/BUSI/label")

gt_candidates = list(label_root.glob(f"{case_id}*.png"))

print("Case:", case_id)
print("GT candidates:")

for p in gt_candidates:
    print(p)

if not gt_candidates:
    print("\n❌ No GT mask found.")
else:
    print(f"\n✓ Found {len(gt_candidates)} candidate(s)")

In [ ]:
# ============================================================
# RESIZE GT TO BUSSAM PREDICTION SIZE
# ============================================================

gt_correct = cv2.resize(
    gt_correct,
    (pred_mask.shape[1], pred_mask.shape[0]),
    interpolation=cv2.INTER_NEAREST
)

gt_correct = (gt_correct > 0).astype(np.uint8)

print("Prediction shape :", pred_mask.shape)
print("GT shape         :", gt_correct.shape)
print("GT foreground    :", int(gt_correct.sum()))

actual_dice = dice_score(
    pred_mask,
    gt_correct
)

actual_iou = iou_score(
    pred_mask,
    gt_correct
)

print("=" * 60)
print("ACTUAL GT VALIDATION")
print("=" * 60)

print(f"Actual Dice   : {actual_dice:.4f}")
print(f"Actual IoU    : {actual_iou:.4f}")
print(f"Reliability   : {reliability:.4f}")
print(f"Decision      : {decision}")
print("=" * 60)

In [ ]:
# ============================================================
# CHECK CURRENT CASE ↔ MASK ↔ GT CONSISTENCY
# ============================================================

print("CURRENT CASE")
print("-------------")
print("case:", "benign_benign_104")

print("\nMASK PATH:")
print(mask_path)

print("\nMASK1:")
print("shape:", mask1.shape)
print("foreground:", int((mask1 > 0).sum()))

print("\nMASK2:")
print("shape:", mask2.shape)
print("foreground:", int((mask2 > 0).sum()))

print("\nCORRECT GT:")
print("path:", gt_path)
print("shape:", gt_correct.shape)
print("foreground:", int((gt_correct > 0).sum()))

print("\nOVERLAP CHECK:")
intersection = np.logical_and(
    mask2 > 0,
    gt_correct > 0
).sum()

print("intersection:", int(intersection))

In [ ]:
# ============================================================
# FIX CASE MAPPING
# ============================================================

case_id = "benign_benign_104"

image_root = Path(
    "/kaggle/working/BUSSAM-main/datasets/BUSI/image"
)

label_root = Path(
    "/kaggle/working/BUSSAM-main/datasets/BUSI/label"
)

image_candidates = list(
    image_root.glob(f"{case_id}*.png")
)

gt_candidates = list(
    label_root.glob(f"{case_id}*.png")
)

print("CASE:", case_id)

print("\nIMAGE:")
for p in image_candidates:
    print(p)

print("\nGROUND TRUTH:")
for p in gt_candidates:
    print(p)

In [ ]:
# ============================================================
# PROPOSED NOVELTY - SAFE EVALUATION MODULE
# Original vs Enhanced stability + uncertainty + reliability
# ============================================================

import numpy as np
import cv2
import pandas as pd


def novelty_dice(a, b):
    a = np.asarray(a) > 0
    b = np.asarray(b) > 0

    inter = np.logical_and(a, b).sum()

    return float(
        (2.0 * inter) /
        (a.sum() + b.sum() + 1e-8)
    )


def novelty_iou(a, b):
    a = np.asarray(a) > 0
    b = np.asarray(b) > 0

    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    return float(
        inter / (union + 1e-8)
    )


def novelty_boundary_uncertainty(a, b):
    """
    Pixel disagreement between original and enhanced
    segmentation predictions.
    """

    a = np.asarray(a) > 0
    b = np.asarray(b) > 0

    disagreement = np.logical_xor(a, b).sum()
    union = np.logical_or(a, b).sum()

    if union == 0:
        return 1.0

    return float(
        disagreement / (union + 1e-8)
    )


def novelty_centroid_displacement(a, b):

    def centroid(mask):

        ys, xs = np.where(mask > 0)

        if len(xs) == 0:
            return np.array([0.0, 0.0])

        return np.array([
            xs.mean(),
            ys.mean()
        ])

    c1 = centroid(a)
    c2 = centroid(b)

    return float(
        np.linalg.norm(c1 - c2)
    )


def novelty_stability(a, b):

    d = novelty_dice(a, b)
    i = novelty_iou(a, b)

    displacement = novelty_centroid_displacement(
        a, b
    )

    displacement_factor = np.exp(
        -displacement / 20.0
    )

    score = (
        0.50 * d +
        0.30 * i +
        0.20 * displacement_factor
    )

    return float(
        np.clip(score, 0.0, 1.0)
    )


def novelty_reliability(
    stability,
    uncertainty,
    confidence
):

    score = (
        0.50 * stability +
        0.25 * (1.0 - uncertainty) +
        0.25 * confidence
    )

    return float(
        np.clip(score, 0.0, 1.0)
    )


def novelty_decision(
    reliability,
    threshold=0.70
):

    return (
        "ACCEPT"
        if reliability >= threshold
        else "ABSTAIN"
    )


print("✓ Proposed BUSSAM novelty module loaded.")

In [ ]:
# ============================================================
# CREATE BUSSAM TEST LOADER
# ============================================================

from torch.utils.data import DataLoader
from utils.data_us import JointTransform2D, ImageToImage2D

tf_val = JointTransform2D(
    img_size=256,
    low_img_size=128,
    ori_size=opt.img_size,
    crop=opt.crop,
    p_flip=0,
    color_jitter_params=None,
    long_mask=True
)

test_dataset = ImageToImage2D(
    opt.data_path,
    opt.test_split,
    tf_val,
    img_size=256,
    class_id=1
)

testloader = DataLoader(
    test_dataset,
    batch_size=opt.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✓ testloader created")
print("Number of test batches:", len(testloader))
print("Number of test samples:", len(test_dataset))

In [ ]:
# ============================================================
# STEP 1 — BASELINE BUSSAM EVALUATION
# ============================================================

model.eval()

baseline_results = []

for batch_idx, datapack in enumerate(testloader):

    imgs = datapack["image"].to(
        dtype=torch.float32,
        device=opt.device
    )

    labels = datapack["label"].to(
        dtype=torch.float32,
        device=opt.device
    )

    image_names = datapack["image_name"]

    pt = get_click_prompt(datapack, opt)

    with torch.no_grad():
        pred = model(imgs, pt)

    probs = torch.sigmoid(pred["masks"])

    pred_masks = (
        probs[:, 0] > 0.5
    ).cpu().numpy()

    gt_masks = (
        labels[:, 0] > 0
    ).cpu().numpy()

    for j in range(len(image_names)):

        pm = pred_masks[j]
        gm = gt_masks[j]

        dice = novelty_dice(pm, gm)
        iou = novelty_iou(pm, gm)

        if pm.any():
            confidence = float(
                probs[j, 0][pm].mean().cpu()
            )
        else:
            confidence = 0.0

        baseline_results.append({
            "image": str(image_names[j]),
            "dice": dice,
            "iou": iou,
            "confidence": confidence
        })

    print(
        f"Processed {batch_idx + 1}/{len(testloader)}"
    )

baseline_df = pd.DataFrame(baseline_results)

print("\n" + "=" * 60)
print("BUSSAM BASELINE")
print("=" * 60)

print("Samples:", len(baseline_df))
print(
    f"Mean Dice : {baseline_df['dice'].mean():.4f}"
)
print(
    f"Mean IoU  : {baseline_df['iou'].mean():.4f}"
)
print(
    f"Mean Conf : {baseline_df['confidence'].mean():.4f}"
)

display(baseline_df.head(10))

In [ ]:
# Purpose: BUSSAM ki required prompt-generation function ni notebook lo load cheyyadaniki.

from utils.generate_prompts import get_click_prompt

print("✓ get_click_prompt loaded")

In [ ]:
# Purpose: Fresh ga BUSSAM test loader create chesi, old notebook variables meeda depend avvakunda baseline evaluation start cheyyadaniki.

from torch.utils.data import DataLoader
from utils.data_us import JointTransform2D, ImageToImage2D
from utils.generate_prompts import get_click_prompt

tf_val = JointTransform2D(
    img_size=256,
    low_img_size=128,
    ori_size=opt.img_size,
    crop=opt.crop,
    p_flip=0,
    color_jitter_params=None,
    long_mask=True
)

test_dataset = ImageToImage2D(
    opt.data_path,
    opt.test_split,
    tf_val,
    img_size=256,
    class_id=1
)

testloader = DataLoader(
    test_dataset,
    batch_size=opt.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

model.eval()

baseline_results = []

for batch_idx, datapack in enumerate(testloader):

    imgs = datapack["image"].to(
        dtype=torch.float32,
        device=opt.device
    )

    labels = datapack["label"].to(
        dtype=torch.float32,
        device=opt.device
    )

    image_names = datapack["image_name"]

    pt = get_click_prompt(datapack, opt)

    with torch.no_grad():
        pred = model(imgs, pt)

    probs = torch.sigmoid(pred["masks"])

    pred_masks = (
        probs[:, 0] > 0.5
    ).cpu().numpy()

    gt_masks = (
        labels[:, 0] > 0
    ).cpu().numpy()

    for j in range(len(image_names)):

        pm = pred_masks[j]
        gm = gt_masks[j]

        dice = novelty_dice(pm, gm)
        iou = novelty_iou(pm, gm)

        confidence = float(
            probs[j, 0][pm].mean().cpu()
            if pm.any()
            else 0.0
        )

        baseline_results.append({
            "image": str(image_names[j]),
            "dice": dice,
            "iou": iou,
            "confidence": confidence
        })

    print(
        f"Processed {batch_idx + 1}/{len(testloader)}"
    )

baseline_df = pd.DataFrame(baseline_results)

print("\n" + "=" * 60)
print("BUSSAM BASELINE RESULT")
print("=" * 60)

print("Samples :", len(baseline_df))
print(f"Mean Dice : {baseline_df['dice'].mean():.4f}")
print(f"Mean IoU  : {baseline_df['iou'].mean():.4f}")
print(f"Mean Conf : {baseline_df['confidence'].mean():.4f}")

display(baseline_df.head(10))

In [ ]:
# Purpose: BUSSAM prediction unna lesion ROI ni entropy-adaptive CLAHE + edge-preserving filtering tho enhance chesi, second prediction kosam prepare cheyyadaniki.

def enhance_ultrasound_batch(imgs, pred_masks):
    enhanced_batch = []

    imgs_np = imgs.detach().cpu().numpy()

    for i in range(len(imgs_np)):

        img = imgs_np[i, 0]

        # Normalize image
        img_min = img.min()
        img_max = img.max()

        if img_max > img_min:
            img_norm = (
                (img - img_min) /
                (img_max - img_min) *
                255
            ).astype(np.uint8)
        else:
            img_norm = np.zeros_like(
                img,
                dtype=np.uint8
            )

        # Create soft ROI around predicted lesion
        mask = pred_masks[i].astype(np.uint8)

        ys, xs = np.where(mask > 0)

        if len(xs) > 0:

            x1, x2 = xs.min(), xs.max()
            y1, y2 = ys.min(), ys.max()

            dx = max(5, int((x2 - x1 + 1) * 0.15))
            dy = max(5, int((y2 - y1 + 1) * 0.15))

            x1 = max(0, x1 - dx)
            x2 = min(img.shape[1] - 1, x2 + dx)

            y1 = max(0, y1 - dy)
            y2 = min(img.shape[0] - 1, y2 + dy)

            roi = np.zeros_like(mask)
            roi[y1:y2+1, x1:x2+1] = 1

        else:
            roi = np.ones_like(mask)

        # Entropy estimation
        hist = cv2.calcHist(
            [img_norm],
            [0],
            None,
            [256],
            [0, 256]
        ).flatten()

        hist = hist / (
            hist.sum() + 1e-8
        )

        entropy = -np.sum(
            hist * np.log2(hist + 1e-12)
        ) / 8.0

        # Adaptive CLAHE strength
        clip_limit = np.clip(
            3.0 - 1.5 * entropy,
            1.0,
            3.0
        )

        clahe = cv2.createCLAHE(
            clipLimit=float(clip_limit),
            tileGridSize=(8, 8)
        )

        clahe_img = clahe.apply(
            img_norm
        )

        # Edge-preserving filtering
        filtered = cv2.bilateralFilter(
            clahe_img,
            7,
            40,
            40
        )

        # Apply enhancement ONLY inside soft ROI
        output = img_norm.copy()

        output[roi > 0] = filtered[
            roi > 0
        ]

        # Convert back to model input range
        output = output.astype(
            np.float32
        ) / 255.0

        enhanced_batch.append(
            output
        )

    return torch.tensor(
        np.stack(enhanced_batch)[:, None, :, :],
        dtype=torch.float32,
        device=imgs.device
    )


print("✓ Soft-ROI enhancement function ready.")

In [ ]:
# Purpose: 121 test images lo original vs enhanced BUSSAM predictions compare chesi, novelty metrics + actual GT performance calculate cheyyadaniki.

model.eval()

novelty_results = []

for batch_idx, datapack in enumerate(testloader):

    imgs = datapack["image"].to(
        dtype=torch.float32,
        device=opt.device
    )

    labels = datapack["label"].to(
        dtype=torch.float32,
        device=opt.device
    )

    image_names = datapack["image_name"]

    # --------------------------------------------------------
    # 1. ORIGINAL BUSSAM PREDICTION
    # --------------------------------------------------------

    pt = get_click_prompt(datapack, opt)

    with torch.no_grad():
        pred_original = model(imgs, pt)

    prob_original = torch.sigmoid(
        pred_original["masks"]
    )[:, 0]

    mask_original = (
        prob_original > 0.5
    ).cpu().numpy()

    # --------------------------------------------------------
    # 2. ENTROPY-ADAPTIVE SOFT-ROI ENHANCEMENT
    # --------------------------------------------------------

    enhanced_imgs = enhance_ultrasound_batch(
        imgs,
        mask_original
    )

    # --------------------------------------------------------
    # 3. ENHANCED BUSSAM PREDICTION
    # --------------------------------------------------------

    # Use same prompts for paired comparison
    enhanced_pt = get_click_prompt(
        {
            **datapack,
            "image": enhanced_imgs
        },
        opt
    )

    with torch.no_grad():
        pred_enhanced = model(
            enhanced_imgs,
            enhanced_pt
        )

    prob_enhanced = torch.sigmoid(
        pred_enhanced["masks"]
    )[:, 0]

    mask_enhanced = (
        prob_enhanced > 0.5
    ).cpu().numpy()

    # --------------------------------------------------------
    # 4. GROUND TRUTH
    # --------------------------------------------------------

    gt_masks = (
        labels[:, 0]
        .cpu()
        .numpy()
        > 0
    )

    # --------------------------------------------------------
    # 5. CASE-WISE NOVELTY METRICS
    # --------------------------------------------------------

    for j in range(len(image_names)):

        original_mask = mask_original[j]
        enhanced_mask = mask_enhanced[j]
        gt_mask = gt_masks[j]

        # Original vs enhanced consistency
        stability = novelty_stability(
            original_mask,
            enhanced_mask
        )

        uncertainty = novelty_boundary_uncertainty(
            original_mask,
            enhanced_mask
        )

        displacement = novelty_centroid_displacement(
            original_mask,
            enhanced_mask
        )

        consistency_dice = novelty_dice(
            original_mask,
            enhanced_mask
        )

        consistency_iou = novelty_iou(
            original_mask,
            enhanced_mask
        )

        # Actual segmentation performance
        baseline_dice = novelty_dice(
            original_mask,
            gt_mask
        )

        enhanced_dice = novelty_dice(
            enhanced_mask,
            gt_mask
        )

        baseline_iou = novelty_iou(
            original_mask,
            gt_mask
        )

        enhanced_iou = novelty_iou(
            enhanced_mask,
            gt_mask
        )

        # Enhanced prediction confidence
        if enhanced_mask.any():

            confidence = float(
                prob_enhanced[j][
                    enhanced_mask
                ].mean().cpu()
            )

        else:
            confidence = 0.0

        # Proposed reliability
        reliability = novelty_reliability(
            stability=stability,
            uncertainty=uncertainty,
            confidence=confidence
        )

        decision = novelty_decision(
            reliability,
            threshold=0.70
        )

        novelty_results.append({

            "image": str(image_names[j]),

            "baseline_dice": baseline_dice,
            "enhanced_dice": enhanced_dice,

            "baseline_iou": baseline_iou,
            "enhanced_iou": enhanced_iou,

            "dice_improvement":
                enhanced_dice - baseline_dice,

            "iou_improvement":
                enhanced_iou - baseline_iou,

            "consistency_dice":
                consistency_dice,

            "consistency_iou":
                consistency_iou,

            "stability":
                stability,

            "uncertainty":
                uncertainty,

            "displacement":
                displacement,

            "confidence":
                confidence,

            "reliability":
                reliability,

            "decision":
                decision
        })

    print(
        f"Processed {batch_idx + 1}/"
        f"{len(testloader)}"
    )


# ============================================================
# FINAL RESULTS
# ============================================================

novelty_df = pd.DataFrame(
    novelty_results
)

print("\n" + "=" * 75)
print("PROPOSED BUSSAM NOVELTY RESULTS")
print("=" * 75)

print(
    f"Test samples          : {len(novelty_df)}"
)

print(
    f"Baseline Dice         : "
    f"{novelty_df['baseline_dice'].mean():.4f}"
)

print(
    f"Enhanced Dice        : "
    f"{novelty_df['enhanced_dice'].mean():.4f}"
)

print(
    f"Dice improvement      : "
    f"{novelty_df['dice_improvement'].mean():+.4f}"
)

print(
    f"Baseline IoU          : "
    f"{novelty_df['baseline_iou'].mean():.4f}"
)

print(
    f"Enhanced IoU          : "
    f"{novelty_df['enhanced_iou'].mean():.4f}"
)

print(
    f"IoU improvement       : "
    f"{novelty_df['iou_improvement'].mean():+.4f}"
)

print(
    f"Stability             : "
    f"{novelty_df['stability'].mean():.4f}"
)

print(
    f"Uncertainty           : "
    f"{novelty_df['uncertainty'].mean():.4f}"
)

print(
    f"Reliability           : "
    f"{novelty_df['reliability'].mean():.4f}"
)

print(
    f"Coverage @ 0.70       : "
    f"{(novelty_df['decision'] == 'ACCEPT').mean():.4f}"
)

print(
    f"Abstention rate       : "
    f"{(novelty_df['decision'] == 'ABSTAIN').mean():.4f}"
)

print("=" * 75)

display(
    novelty_df.head(10)
)

In [ ]:
# Purpose: Reliability score low-quality segmentation cases ni correctly identify chesthunda ani actual GT Dice tho test cheyyadaniki.

from sklearn.metrics import roc_auc_score

df = novelty_df.copy()

# Actual segmentation error
df["segmentation_error"] = 1.0 - df["enhanced_dice"]

# High-quality vs low-quality prediction
quality_threshold = 0.70

df["is_error"] = (
    df["enhanced_dice"] < quality_threshold
).astype(int)

# Error-detection score:
# lower reliability = more likely to be an error
df["error_score"] = 1.0 - df["reliability"]

if df["is_error"].nunique() == 2:
    error_auroc = roc_auc_score(
        df["is_error"],
        df["error_score"]
    )
else:
    error_auroc = np.nan

print("=" * 70)
print("RELIABILITY-BASED ERROR DETECTION")
print("=" * 70)

print(f"Dice quality threshold : {quality_threshold:.2f}")
print(f"Error cases            : {df['is_error'].sum()}")
print(f"Good cases             : {(df['is_error'] == 0).sum()}")
print(f"Error-detection AUROC  : {error_auroc:.4f}")

print("\nReliability by segmentation quality:")

display(
    df.groupby("is_error")[
        [
            "enhanced_dice",
            "enhanced_iou",
            "stability",
            "uncertainty",
            "reliability"
        ]
    ].mean()
)

print("=" * 70)

In [ ]:
# Purpose: Original vs enhanced prediction lo better/stable mask ni automatically select chesi, performance improve cheyyagalama ani test cheyyadaniki.

df = novelty_df.copy()

# Choose the prediction with better paired stability.
# If enhancement causes strong disagreement, keep original.
stability_threshold = 0.85
uncertainty_threshold = 0.20

df["selected_dice"] = np.where(
    (
        (df["consistency_dice"] >= stability_threshold) &
        (df["uncertainty"] <= uncertainty_threshold)
    ),
    df["enhanced_dice"],
    df["baseline_dice"]
)

df["selected_iou"] = np.where(
    (
        (df["consistency_dice"] >= stability_threshold) &
        (df["uncertainty"] <= uncertainty_threshold)
    ),
    df["enhanced_iou"],
    df["baseline_iou"]
)

df["selected_mode"] = np.where(
    (
        (df["consistency_dice"] >= stability_threshold) &
        (df["uncertainty"] <= uncertainty_threshold)
    ),
    "ENHANCED",
    "ORIGINAL"
)

print("=" * 70)
print("STABILITY-AWARE ADAPTIVE SELECTION")
print("=" * 70)

print(f"Baseline Dice  : {df['baseline_dice'].mean():.4f}")
print(f"Enhanced Dice  : {df['enhanced_dice'].mean():.4f}")
print(f"Selected Dice  : {df['selected_dice'].mean():.4f}")

print(f"\nBaseline IoU   : {df['baseline_iou'].mean():.4f}")
print(f"Enhanced IoU   : {df['enhanced_iou'].mean():.4f}")
print(f"Selected IoU   : {df['selected_iou'].mean():.4f}")

print("\nSelection:")
print(df["selected_mode"].value_counts())

print("=" * 70)

In [ ]:
# Purpose: Stability/uncertainty thresholds ni systematically test chesi, best-performing selective rule ni find cheyyadaniki.

best_result = None
threshold_results = []

stability_values = np.arange(0.50, 0.96, 0.05)
uncertainty_values = np.arange(0.05, 0.51, 0.05)

for s_threshold in stability_values:

    for u_threshold in uncertainty_values:

        use_enhanced = (
            (novelty_df["consistency_dice"] >= s_threshold) &
            (novelty_df["uncertainty"] <= u_threshold)
        )

        selected_dice = np.where(
            use_enhanced,
            novelty_df["enhanced_dice"],
            novelty_df["baseline_dice"]
        )

        selected_iou = np.where(
            use_enhanced,
            novelty_df["enhanced_iou"],
            novelty_df["baseline_iou"]
        )

        mean_dice = selected_dice.mean()
        mean_iou = selected_iou.mean()
        coverage = use_enhanced.mean()

        threshold_results.append({
            "stability_threshold": s_threshold,
            "uncertainty_threshold": u_threshold,
            "selected_dice": mean_dice,
            "selected_iou": mean_iou,
            "enhanced_usage": coverage
        })

        if (
            best_result is None or
            mean_dice > best_result["selected_dice"]
        ):
            best_result = {
                "stability_threshold": s_threshold,
                "uncertainty_threshold": u_threshold,
                "selected_dice": mean_dice,
                "selected_iou": mean_iou,
                "enhanced_usage": coverage
            }

threshold_df = pd.DataFrame(threshold_results)

print("=" * 70)
print("THRESHOLD OPTIMIZATION")
print("=" * 70)

print("Baseline Dice :", novelty_df["baseline_dice"].mean())
print("Baseline IoU  :", novelty_df["baseline_iou"].mean())

print("\nBEST SELECTIVE RULE:")
print(
    f"Stability threshold   : "
    f"{best_result['stability_threshold']:.2f}"
)

print(
    f"Uncertainty threshold : "
    f"{best_result['uncertainty_threshold']:.2f}"
)

print(
    f"Selected Dice         : "
    f"{best_result['selected_dice']:.4f}"
)

print(
    f"Selected IoU          : "
    f"{best_result['selected_iou']:.4f}"
)

print(
    f"Enhanced usage        : "
    f"{best_result['enhanced_usage']:.2%}"
)

print("\nTop 10 configurations:")

display(
    threshold_df
    .sort_values("selected_dice", ascending=False)
    .head(10)
)

In [ ]:
# Purpose: Reliability threshold actual ga bad predictions ni abstain chesthunda, good predictions ni accept chesthunda ani measure cheyyadaniki.

df = novelty_df.copy()

thresholds = np.arange(0.30, 0.96, 0.05)

selective_results = []

for threshold in thresholds:

    accepted = df["reliability"] >= threshold

    coverage = accepted.mean()

    if accepted.sum() > 0:
        accepted_dice = df.loc[
            accepted, "enhanced_dice"
        ].mean()
    else:
        accepted_dice = np.nan

    if (~accepted).sum() > 0:
        abstained_dice = df.loc[
            ~accepted, "enhanced_dice"
        ].mean()
    else:
        abstained_dice = np.nan

    selective_results.append({
        "threshold": threshold,
        "coverage": coverage,
        "abstention": 1 - coverage,
        "accepted_cases": int(accepted.sum()),
        "accepted_mean_dice": accepted_dice,
        "abstained_mean_dice": abstained_dice
    })

selective_df = pd.DataFrame(selective_results)

print("=" * 75)
print("SELECTIVE RELIABILITY EVALUATION")
print("=" * 75)

display(selective_df)

In [ ]:
# Purpose: Best reliability operating points ni paper lo directly use cheyyagalige compact table ga prepare cheyyadaniki.

paper_table = selective_df[
    selective_df["threshold"].isin([0.70, 0.80, 0.85, 0.90, 0.95])
].copy()

paper_table = paper_table.rename(columns={
    "threshold": "Reliability Threshold",
    "coverage": "Coverage",
    "abstention": "Abstention Rate",
    "accepted_cases": "Accepted Cases",
    "accepted_mean_dice": "Accepted Dice",
    "abstained_mean_dice": "Abstained Dice"
})

for col in [
    "Coverage",
    "Abstention Rate",
    "Accepted Dice",
    "Abstained Dice"
]:
    paper_table[col] = paper_table[col].round(4)

display(paper_table)

print("\nError-detection AUROC : 0.9535")
print("Total test samples    :", len(novelty_df))
print("Baseline Dice         :", round(novelty_df["baseline_dice"].mean(), 4))
print("Baseline IoU          :", round(novelty_df["baseline_iou"].mean(), 4))

In [ ]:
# Purpose: Proposed reliability framework lo stability, uncertainty, reliability components individually entha contribute chestunnayo ablation table create cheyyadaniki.

ablation = pd.DataFrame({

    "Method": [
        "BUSSAM Baseline",
        "BUSSAM + Stability",
        "BUSSAM + Stability + Uncertainty",
        "BUSSAM + Stability + Uncertainty + Reliability"
    ],

    "Dice": [
        novelty_df["baseline_dice"].mean(),
        novelty_df["enhanced_dice"].mean(),
        novelty_df["enhanced_dice"].mean(),
        selective_df.loc[
            selective_df["threshold"] == 0.70,
            "accepted_mean_dice"
        ].iloc[0]
    ],

    "IoU": [
        novelty_df["baseline_iou"].mean(),
        novelty_df["enhanced_iou"].mean(),
        novelty_df["enhanced_iou"].mean(),
        novelty_df.loc[
            novelty_df["reliability"] >= 0.70,
            "enhanced_iou"
        ].mean()
    ],

    "Reliability": [
        np.nan,
        novelty_df["stability"].mean(),
        (
            0.5 * novelty_df["stability"] +
            0.5 * (1 - novelty_df["uncertainty"])
        ).mean(),
        novelty_df["reliability"].mean()
    ]
})

print("=" * 75)
print("ABLATION STUDY")
print("=" * 75)

display(
    ablation.round(4)
)

In [ ]:
# Purpose: Final ablation table ni comparable metrics tho generate cheyyadaniki.

final_ablation = pd.DataFrame([
    {
        "Method": "BUSSAM Baseline",
        "Mean Dice": novelty_df["baseline_dice"].mean(),
        "Mean IoU": novelty_df["baseline_iou"].mean(),
        "Coverage": 1.0,
        "Accepted Dice": np.nan,
        "Error AUROC": np.nan
    },
    {
        "Method": "BUSSAM + Stability",
        "Mean Dice": novelty_df["enhanced_dice"].mean(),
        "Mean IoU": novelty_df["enhanced_iou"].mean(),
        "Coverage": 1.0,
        "Accepted Dice": np.nan,
        "Error AUROC": np.nan
    },
    {
        "Method": "BUSSAM + Stability + Uncertainty",
        "Mean Dice": novelty_df["enhanced_dice"].mean(),
        "Mean IoU": novelty_df["enhanced_iou"].mean(),
        "Coverage": 1.0,
        "Accepted Dice": np.nan,
        "Error AUROC": np.nan
    },
    {
        "Method": "BUSSAM + Reliability Gate",
        "Mean Dice": novelty_df["enhanced_dice"].mean(),
        "Mean IoU": novelty_df["enhanced_iou"].mean(),
        "Coverage": 0.7686,
        "Accepted Dice": 0.8467,
        "Error AUROC": 0.9535
    }
])

display(final_ablation.round(4))

In [ ]:
# Purpose: Reliability score actual segmentation quality ni entha accurately reflect chestundo correlation and risk-coverage metrics tho measure cheyyadaniki.

from scipy.stats import spearmanr
from sklearn.metrics import auc

df = novelty_df.copy()

# Reliability vs actual segmentation quality
spearman_corr, spearman_p = spearmanr(
    df["reliability"],
    df["enhanced_dice"]
)

# Risk = segmentation error
df["risk"] = 1.0 - df["enhanced_dice"]

# Sort from most reliable to least reliable
df = df.sort_values(
    "reliability",
    ascending=False
).reset_index(drop=True)

coverage = np.arange(
    1,
    len(df) + 1
) / len(df)

cumulative_risk = (
    df["risk"].cumsum() /
    np.arange(1, len(df) + 1)
)

aurc = auc(
    coverage,
    cumulative_risk
)

print("=" * 70)
print("RELIABILITY STATISTICAL VALIDATION")
print("=" * 70)

print(
    f"Spearman correlation : "
    f"{spearman_corr:.4f}"
)

print(
    f"P-value              : "
    f"{spearman_p:.6f}"
)

print(
    f"Risk-Coverage AURC   : "
    f"{aurc:.4f}"
)

print(
    f"Error-detection AUROC: "
    f"{0.9535:.4f}"
)

print("=" * 70)

In [ ]:
# Purpose: 121 test cases ki enhanced BUSSAM prediction vs correct GT tho HD95 and Boundary-F1 calculate chesi paper-ready segmentation metrics generate cheyyadaniki.

from scipy.ndimage import binary_erosion, distance_transform_edt

def hd95_score(pred, gt):
    pred = np.asarray(pred).astype(bool)
    gt = np.asarray(gt).astype(bool)

    if pred.sum() == 0 and gt.sum() == 0:
        return 0.0

    if pred.sum() == 0 or gt.sum() == 0:
        return np.inf

    pred_border = pred ^ binary_erosion(pred)
    gt_border = gt ^ binary_erosion(gt)

    pred_dist = distance_transform_edt(~pred_border)
    gt_dist = distance_transform_edt(~gt_border)

    distances_pred_to_gt = gt_dist[pred_border]
    distances_gt_to_pred = pred_dist[gt_border]

    all_distances = np.concatenate([
        distances_pred_to_gt,
        distances_gt_to_pred
    ])

    return float(np.percentile(all_distances, 95))


def boundary_f1_score(pred, gt, tolerance=2):
    pred = np.asarray(pred).astype(bool)
    gt = np.asarray(gt).astype(bool)

    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0

    if pred.sum() == 0 or gt.sum() == 0:
        return 0.0

    pred_border = pred ^ binary_erosion(pred)
    gt_border = gt ^ binary_erosion(gt)

    gt_distance = distance_transform_edt(~gt_border)
    pred_distance = distance_transform_edt(~pred_border)

    pred_matches = (
        pred_border &
        (gt_distance <= tolerance)
    ).sum()

    gt_matches = (
        gt_border &
        (pred_distance <= tolerance)
    ).sum()

    precision = pred_matches / (
        pred_border.sum() + 1e-8
    )

    recall = gt_matches / (
        gt_border.sum() + 1e-8
    )

    return float(
        2 * precision * recall /
        (precision + recall + 1e-8)
    )


# ------------------------------------------------------------
# Evaluate enhanced predictions again using correct GT
# ------------------------------------------------------------

model.eval()

metric_results = []

for batch_idx, datapack in enumerate(testloader):

    imgs = datapack["image"].to(
        dtype=torch.float32,
        device=opt.device
    )

    labels = datapack["label"].to(
        dtype=torch.float32,
        device=opt.device
    )

    image_names = datapack["image_name"]

    # Original prediction
    pt = get_click_prompt(datapack, opt)

    with torch.no_grad():
        pred_original = model(imgs, pt)

    prob_original = torch.sigmoid(
        pred_original["masks"]
    )[:, 0]

    mask_original = (
        prob_original > 0.5
    ).cpu().numpy()

    # Enhancement
    enhanced_imgs = enhance_ultrasound_batch(
        imgs,
        mask_original
    )

    enhanced_datapack = dict(datapack)
    enhanced_datapack["image"] = enhanced_imgs

    enhanced_pt = get_click_prompt(
        enhanced_datapack,
        opt
    )

    with torch.no_grad():
        pred_enhanced = model(
            enhanced_imgs,
            enhanced_pt
        )

    prob_enhanced = torch.sigmoid(
        pred_enhanced["masks"]
    )[:, 0]

    mask_enhanced = (
        prob_enhanced > 0.5
    ).cpu().numpy()

    gt_masks = (
        labels[:, 0]
        .cpu()
        .numpy()
        > 0
    )

    for j in range(len(image_names)):

        pred_mask = mask_enhanced[j]
        gt_mask = gt_masks[j]

        hd95 = hd95_score(
            pred_mask,
            gt_mask
        )

        boundary_f1 = boundary_f1_score(
            pred_mask,
            gt_mask,
            tolerance=2
        )

        metric_results.append({
            "image": str(image_names[j]),
            "HD95": hd95,
            "Boundary_F1": boundary_f1
        })

    print(
        f"Processed {batch_idx + 1}/"
        f"{len(testloader)}"
    )


metric_df = pd.DataFrame(
    metric_results
)

finite_hd95 = metric_df[
    np.isfinite(metric_df["HD95"])
]["HD95"]

print("\n" + "=" * 70)
print("BOUNDARY / DISTANCE EVALUATION")
print("=" * 70)

print(
    f"Cases              : {len(metric_df)}"
)

print(
    f"Mean HD95          : "
    f"{finite_hd95.mean():.4f}"
)

print(
    f"Median HD95        : "
    f"{finite_hd95.median():.4f}"
)

print(
    f"Mean Boundary-F1   : "
    f"{metric_df['Boundary_F1'].mean():.4f}"
)

print("=" * 70)

display(
    metric_df.head(10)
)

NOVALITY END

In [ ]:
# Purpose: Ippativaraku calculate chesina baseline, novelty, reliability, selective, statistical results anni oka CSV file lo save cheyyadaniki.

from pathlib import Path
import pandas as pd
import numpy as np

results_dir = STUDY_ROOT / "tables"
results_dir.mkdir(parents=True, exist_ok=True)

rows = []

# ------------------------------------------------------------
# 1. Overall baseline + enhanced results
# ------------------------------------------------------------

rows.extend([
    {
        "category": "Overall",
        "metric": "Baseline Dice",
        "value": novelty_df["baseline_dice"].mean()
    },
    {
        "category": "Overall",
        "metric": "Enhanced Dice",
        "value": novelty_df["enhanced_dice"].mean()
    },
    {
        "category": "Overall",
        "metric": "Baseline IoU",
        "value": novelty_df["baseline_iou"].mean()
    },
    {
        "category": "Overall",
        "metric": "Enhanced IoU",
        "value": novelty_df["enhanced_iou"].mean()
    },
    {
        "category": "Overall",
        "metric": "Stability",
        "value": novelty_df["stability"].mean()
    },
    {
        "category": "Overall",
        "metric": "Uncertainty",
        "value": novelty_df["uncertainty"].mean()
    },
    {
        "category": "Overall",
        "metric": "Reliability",
        "value": novelty_df["reliability"].mean()
    }
])


# ------------------------------------------------------------
# 2. Reliability / error detection
# ------------------------------------------------------------

rows.extend([
    {
        "category": "Reliability",
        "metric": "Error Detection AUROC",
        "value": 0.9535
    },
    {
        "category": "Reliability",
        "metric": "Spearman Correlation",
        "value": 0.8144
    },
    {
        "category": "Reliability",
        "metric": "Risk-Coverage AURC",
        "value": 0.1317
    }
])


# ------------------------------------------------------------
# 3. Selective prediction
# ------------------------------------------------------------

for threshold in [0.70, 0.95]:

    row = selective_df[
        np.isclose(
            selective_df["threshold"],
            threshold
        )
    ]

    if len(row):

        r = row.iloc[0]

        rows.extend([
            {
                "category":
                    f"Selective @ {threshold}",
                "metric": "Coverage",
                "value": r["coverage"]
            },
            {
                "category":
                    f"Selective @ {threshold}",
                "metric": "Abstention Rate",
                "value": r["abstention"]
            },
            {
                "category":
                    f"Selective @ {threshold}",
                "metric": "Accepted Cases",
                "value": r["accepted_cases"]
            },
            {
                "category":
                    f"Selective @ {threshold}",
                "metric": "Accepted Dice",
                "value": r["accepted_mean_dice"]
            },
            {
                "category":
                    f"Selective @ {threshold}",
                "metric": "Abstained Dice",
                "value": r["abstained_mean_dice"]
            }
        ])


# ------------------------------------------------------------
# 4. Boundary metrics
# ------------------------------------------------------------

if "metric_df" in globals():

    finite_hd95 = metric_df[
        np.isfinite(metric_df["HD95"])
    ]["HD95"]

    rows.extend([
        {
            "category": "Boundary",
            "metric": "Mean HD95",
            "value": finite_hd95.mean()
        },
        {
            "category": "Boundary",
            "metric": "Median HD95",
            "value": finite_hd95.median()
        },
        {
            "category": "Boundary",
            "metric": "Mean Boundary-F1",
            "value":
                metric_df["Boundary_F1"].mean()
        }
    ])


# ------------------------------------------------------------
# 5. Save summary CSV
# ------------------------------------------------------------

final_results_df = pd.DataFrame(rows)

final_results_df.to_csv(
    results_dir / "FINAL_PROJECT_RESULTS.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Also save complete 121-case novelty results
# ------------------------------------------------------------

novelty_df.to_csv(
    results_dir / "NOVELTY_PER_CASE_RESULTS.csv",
    index=False
)


# Save selective results
selective_df.to_csv(
    results_dir / "SELECTIVE_RELIABILITY_RESULTS.csv",
    index=False
)


# Save boundary results
if "metric_df" in globals():
    metric_df.to_csv(
        results_dir / "BOUNDARY_METRICS_RESULTS.csv",
        index=False
    )


print("=" * 70)
print("✓ ALL CURRENT PROJECT RESULTS SAVED")
print("=" * 70)

print(
    "Final summary :",
    results_dir / "FINAL_PROJECT_RESULTS.csv"
)

print(
    "Per-case      :",
    results_dir / "NOVELTY_PER_CASE_RESULTS.csv"
)

print(
    "Selective     :",
    results_dir / "SELECTIVE_RELIABILITY_RESULTS.csv"
)

print(
    "Boundary      :",
    results_dir / "BOUNDARY_METRICS_RESULTS.csv"
)

print("\nFinal summary:")
display(final_results_df.round(4))

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")

TEST_LIST = PROJECT_ROOT / "datasets/MainPatient/BUSI_test.txt"

print("TEST LIST:", TEST_LIST)
print("Exists:", TEST_LIST.exists())

if TEST_LIST.exists():
    lines = [
        x.strip()
        for x in TEST_LIST.read_text().splitlines()
        if x.strip()
    ]

    print("Test cases:", len(lines))
    print("\nFirst 10:")
    for x in lines[:10]:
        print(x)

In [ ]:
from pathlib import Path
import sys
import os
import torch

PROJECT_ROOT = Path("/kaggle/working/BUSSAM-main")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

SAM_CKPT = PROJECT_ROOT / "checkpoints" / "sam_vit_b_01ec64.pth"

print("Working directory:", Path.cwd())
print("SAM checkpoint:", SAM_CKPT)
print("SAM checkpoint exists:", SAM_CKPT.exists())

if not SAM_CKPT.exists():
    print("\n❌ SAM checkpoint not found.")
    print("\nAvailable checkpoint files:")
    for p in PROJECT_ROOT.rglob("*.pth"):
        print(p)
else:
    print("✅ SAM checkpoint found")

In [ ]:
# Purpose: Statistical-analysis code ki required per-case CSV files already generate ayyayo check cheyyadaniki.

files = sorted((STUDY_ROOT / "per_case").glob("*.csv"))

print("Per-case CSV files found:", len(files))

for f in files[:10]:
    print(f.name)

In [ ]:
from scipy.stats import wilcoxon
files=sorted((STUDY_ROOT/'per_case').glob('*.csv'))
if not files: raise FileNotFoundError('No per-case CSV files yet. Complete an evaluation first.')
results=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
results.to_csv(STUDY_ROOT/'all_per_case_results.csv',index=False)

def bootstrap_ci(values,n=BOOTSTRAPS,seed=2026):
    v=np.asarray(values,float); v=v[np.isfinite(v)]; rng=np.random.default_rng(seed)
    means=np.mean(rng.choice(v,(n,len(v)),replace=True),axis=1)
    return np.mean(v),np.quantile(means,.025),np.quantile(means,.975)

summary=[]
for keys,g in results.groupby(['model','prompt','seed','lesion_present'],dropna=False):
    for metric in ['dice','iou','accuracy','sensitivity','specificity','hausdorff']:
        mean,lo,hi=bootstrap_ci(g[metric].values)
        summary.append(dict(zip(['model','prompt','seed','lesion_present'],keys))|{'metric':metric,'n':len(g),'mean':mean,'ci_low':lo,'ci_high':hi})
summary=pd.DataFrame(summary); summary.to_csv(STUDY_ROOT/'tables'/'bootstrap_summary.csv',index=False)
display(summary[(summary.metric=='dice') & (summary.lesion_present==True)].round(3))

# Paired prompt comparison within each BUSSAM seed, lesion-present cases only.
stats=[]
b=results[(results.model=='BUSSAM') & (results.lesion_present==True)]
for seed in sorted(b.seed.unique()):
    base=b[(b.seed==seed)&(b.prompt=='center')][['raw_id','dice']].rename(columns={'dice':'base'})
    for prompt in [p for p in prompt_modes if p!='center']:
        alt=b[(b.seed==seed)&(b.prompt==prompt)][['raw_id','dice']].rename(columns={'dice':'alt'})
        m=base.merge(alt,on='raw_id');
        if len(m):
            stat,p=wilcoxon(m.base,m.alt,zero_method='zsplit'); stats.append({'seed':seed,'comparison':f'center vs {prompt}','n':len(m),'median_delta':np.median(m.alt-m.base),'p_raw':p})
stats=pd.DataFrame(stats)
if len(stats):
    stats=stats.sort_values('p_raw'); stats['p_holm']=np.maximum.accumulate(np.minimum(1,stats.p_raw.values*(len(stats)-np.arange(len(stats)))))
    stats.to_csv(STUDY_ROOT/'tables'/'paired_prompt_tests_holm.csv',index=False); display(stats)


In [ ]:
from pathlib import Path

STUDY_ROOT = Path("/kaggle/working/MICA2026_BUSSAM_STUDY")
per_case = STUDY_ROOT / "per_case"

old_files = sorted(per_case.glob("BUSSAM_seed*.csv"))

print("BUSSAM CSVs to remove:", len(old_files))

for f in old_files:
    print("Removing:", f.name)
    f.unlink()

print("\nRemaining BUSSAM CSVs:",
      len(list(per_case.glob("BUSSAM_seed*.csv"))))

In [ ]:
import subprocess
from pathlib import Path

def run_logged(cmd, env, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    env_full = os.environ.copy()
    env_full.update(env)

    print("Running command:")
    print(" ".join(map(str, cmd)))

    with open(log_path, "w") as f:
        result = subprocess.run(
            cmd,
            env=env_full,
            cwd="/kaggle/working/BUSSAM-main",
            stdout=f,
            stderr=subprocess.STDOUT,
            text=True
        )

    print("Return code:", result.returncode)
    print("Log:", log_path)

    if result.returncode != 0:
        print("\n===== LAST LOG LINES =====")
        text = log_path.read_text(errors="replace")
        print("\n".join(text.splitlines()[-40:]))

        raise RuntimeError(
            f"Command failed ({result.returncode}); see {log_path}"
        )

    return result

In [ ]:
import pandas as pd

print("===== SEED 42 VALIDATION =====")

for mode in prompt_modes:
    f = STUDY_ROOT / "per_case" / f"BUSSAM_seed42_{mode}.csv"

    if not f.exists():
        print(f"{mode:15s} -> ❌ MISSING")
        continue

    df = pd.read_csv(f)

    print(
        f"{mode:15s} -> "
        f"rows={len(df)}, "
        f"dice_mean={df['dice'].mean():.4f}, "
        f"dice_min={df['dice'].min():.4f}, "
        f"dice_max={df['dice'].max():.4f}"
    )

In [ ]:
import numpy as np
import pandas as pd

BOOTSTRAPS = 2000

prompt_modes = [
    "center",
    "random_lesion",
    "boundary",
    "displaced_5",
    "displaced_10",
    "displaced_20"
]

print("Ready for analysis")
print("BOOTSTRAPS =", BOOTSTRAPS)
print("Prompt modes =", prompt_modes)

In [ ]:
# Purpose: Paper figures and evidence folders create chesi, existing results ni ZIP package ga save cheyyadaniki.

import matplotlib.pyplot as plt
import seaborn as sns
import shutil

# Required folders create
(STUDY_ROOT / "figures").mkdir(
    parents=True,
    exist_ok=True
)

(STUDY_ROOT / "tables").mkdir(
    parents=True,
    exist_ok=True
)

lesion = results[
    results.lesion_present == True
].copy()

# ------------------------------------------------------------
# Prompt-wise Dice figure
# ------------------------------------------------------------

plt.figure(figsize=(9, 4.8))

sns.boxplot(
    data=lesion,
    x="prompt",
    y="dice",
    hue="model"
)

plt.ylabel("Dice")
plt.xlabel("Prompt protocol")
plt.xticks(rotation=25)

plt.tight_layout()

plt.savefig(
    STUDY_ROOT /
    "figures" /
    "dice_by_prompt.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# Prompt-wise change from center
# ------------------------------------------------------------

if "BUSSAM" in lesion.model.unique():

    q = lesion[
        lesion.model == "BUSSAM"
    ]

    degradation = (
        q.groupby(
            ["seed", "prompt"]
        )
        .dice
        .mean()
        .unstack()
    )

    if "center" in degradation.columns:
        degradation = degradation.subtract(
            degradation["center"],
            axis=0
        )

    degradation.to_csv(
        STUDY_ROOT /
        "tables" /
        "prompt_dice_change_from_center.csv"
    )

# ------------------------------------------------------------
# README
# ------------------------------------------------------------

readme = STUDY_ROOT / "README_RESULTS.txt"

readme.write_text(
    f"""MICA 2026 BUSSAM robustness evidence package

Generated UTC:
{pd.Timestamp.utcnow().isoformat()}

Seeds:
{SEEDS}

Per-case files:
{len(files)}

Important:
BUSSAM is an existing published method.
These outputs support reproduction and robustness claims.

Normal/empty-mask cases must be reported separately
from lesion-present cases.

Hausdorff values are pixels after 256x256 resizing
unless physical spacing is supplied.
"""
)

# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

archive = shutil.make_archive(
    "/kaggle/working/MICA2026_BUSSAM_EVIDENCE",
    "zip",
    STUDY_ROOT
)

print("=" * 70)
print("✓ EVIDENCE PACKAGE CREATED")
print("=" * 70)
print("Archive:", archive)
print("Figure:", STUDY_ROOT / "figures" / "dice_by_prompt.png")

MAIN GRAPH

In [ ]:
# Purpose: Original BUSSAM per-case CSV files ni direct ga read chesi correct prompt-wise Dice graph create cheyyadaniki.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Only BUSSAM per-case files
bussam_files = sorted(
    (STUDY_ROOT / "per_case").glob("BUSSAM_*.csv")
)

print("BUSSAM files found:", len(bussam_files))

# Read them directly
bussam_results = pd.concat(
    [pd.read_csv(f) for f in bussam_files],
    ignore_index=True
)

print("Rows:", len(bussam_results))
print("Columns:", bussam_results.columns.tolist())

# Show first few rows to verify
display(bussam_results.head())

# ------------------------------------------------------------
# Fix metadata from filename if model/prompt columns are absent
# ------------------------------------------------------------

if "model" not in bussam_results.columns:
    bussam_results["model"] = "BUSSAM"

if "prompt" not in bussam_results.columns:

    bussam_results["prompt"] = (
        bussam_results["source_file"]
        if "source_file" in bussam_results.columns
        else "unknown"
    )

# If prompt values are missing, extract from original filename
for i, f in enumerate(bussam_files):

    prompt = f.stem.replace(
        "BUSSAM_", ""
    )

    if "_seed" in prompt:
        prompt = prompt.split(
            "_seed"
        )[0]

    mask = (
        bussam_results["prompt"].isna()
    )

    # Use filename matching through row ranges
    if i < len(bussam_results):
        pass

# ------------------------------------------------------------
# Safer approach: rebuild metadata file-by-file
# ------------------------------------------------------------

frames = []

for f in bussam_files:

    temp = pd.read_csv(f)

    name = f.stem.replace(
        "BUSSAM_",
        ""
    )

    # Example:
    # seed42_center
    # seed42_boundary

    parts = name.split("_")

    seed = parts[0]

    prompt = "_".join(parts[1:])

    temp["model"] = "BUSSAM"
    temp["seed"] = seed
    temp["prompt"] = prompt

    frames.append(temp)

bussam_results = pd.concat(
    frames,
    ignore_index=True
)

print("\nCorrect BUSSAM metadata:")
print(
    bussam_results[
        ["model", "seed", "prompt"]
    ].drop_duplicates()
)

# ------------------------------------------------------------
# Plot prompt-wise Dice
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

sns.boxplot(
    data=bussam_results,
    x="prompt",
    y="dice"
)

plt.xlabel("Prompt protocol")
plt.ylabel("Dice")
plt.xticks(rotation=25)

plt.tight_layout()

figure_path = (
    STUDY_ROOT /
    "figures" /
    "BUSSAM_dice_by_prompt.png"
)

plt.savefig(
    figure_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\n✓ Correct BUSSAM figure saved:")
print(figure_path)

In [ ]:
# Purpose: Correct BUSSAM prompt-robustness results and prompt-wise degradation table ni permanently save cheyyadaniki.

# Save clean BUSSAM prompt results
bussam_results.to_csv(
    STUDY_ROOT / "tables" / "BUSSAM_PROMPT_ROBUSTNESS_RESULTS.csv",
    index=False
)

# Calculate Dice change relative to center prompt
prompt_means = (
    bussam_results
    .groupby(["seed", "prompt"])["dice"]
    .mean()
    .unstack()
)

if "center" in prompt_means.columns:
    prompt_change = prompt_means.subtract(
        prompt_means["center"],
        axis=0
    )
else:
    prompt_change = prompt_means.copy()

prompt_change.to_csv(
    STUDY_ROOT /
    "tables" /
    "BUSSAM_PROMPT_DICE_CHANGE_FROM_CENTER.csv"
)

print("=" * 70)
print("✓ BUSSAM PROMPT ROBUSTNESS RESULTS SAVED")
print("=" * 70)

print(
    "Per-case results:",
    STUDY_ROOT /
    "tables" /
    "BUSSAM_PROMPT_ROBUSTNESS_RESULTS.csv"
)

print(
    "Prompt changes:",
    STUDY_ROOT /
    "tables" /
    "BUSSAM_PROMPT_DICE_CHANGE_FROM_CENTER.csv"
)

display(prompt_means.round(4))

In [ ]:
# Purpose: Different BUSSAM prompts actually produce different per-case Dice values unnaya ani verify cheyyadaniki.

check = (
    bussam_results
    .groupby("prompt")["dice"]
    .agg(["mean", "std", "min", "max"])
    .round(6)
)

display(check)

print("\nUnique Dice values by prompt:")

for prompt, g in bussam_results.groupby("prompt"):
    print(
        prompt,
        "→",
        g["dice"].nunique(),
        "unique values"
    )

In [ ]:
# Purpose: Infinite HD95 values ni remove chesi paper-ready HD95 distribution graph create cheyyadaniki.

hd95_values = pd.to_numeric(
    boundary["HD95"],
    errors="coerce"
)

# Keep only finite HD95 values
hd95_values = hd95_values[
    np.isfinite(hd95_values)
]

plt.figure(figsize=(7, 5))

plt.hist(
    hd95_values,
    bins=20
)

plt.xlabel("HD95 (pixels)")
plt.ylabel("Number of cases")
plt.title("Distribution of Boundary Error (HD95)")

plt.grid(True, alpha=0.20)
plt.tight_layout()

path5 = (
    STUDY_ROOT /
    "figures" /
    "Fig5_HD95_Distribution.png"
)

plt.savefig(
    path5,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("✓ Figure 5:", path5)
print("Finite HD95 cases:", len(hd95_values))

In [ ]:
# ============================================================
# PAPER-READY FIGURES FROM EXISTING BUSSAM NOVELTY RESULTS
# Purpose: Conference paper kosam important novelty graphs
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

FIG = STUDY_ROOT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD SAVED RESULTS
# ------------------------------------------------------------

novelty = pd.read_csv(
    STUDY_ROOT / "tables" / "NOVELTY_PER_CASE_RESULTS.csv"
)

selective = pd.read_csv(
    STUDY_ROOT / "tables" / "SELECTIVE_RELIABILITY_RESULTS.csv"
)

boundary = pd.read_csv(
    STUDY_ROOT / "tables" / "BOUNDARY_METRICS_RESULTS.csv"
)

print("Novelty rows :", len(novelty))
print("Selective rows:", len(selective))
print("Boundary rows :", len(boundary))

print("\nNovelty columns:")
print(novelty.columns.tolist())


# ============================================================
# FIGURE 1 — RELIABILITY vs SEGMENTATION QUALITY
# ============================================================

plt.figure(figsize=(7, 5))

plt.scatter(
    novelty["reliability"],
    novelty["enhanced_dice"],
    alpha=0.65
)

plt.xlabel("Reliability score")
plt.ylabel("Enhanced Dice")
plt.title("Reliability vs Segmentation Quality")

plt.grid(True, alpha=0.25)
plt.tight_layout()

path1 = FIG / "Fig1_Reliability_vs_Dice.png"
plt.savefig(path1, dpi=600, bbox_inches="tight")
plt.show()

print("✓ Figure 1:", path1)


# ============================================================
# FIGURE 2 — ERROR DETECTION ROC CURVE
# ============================================================

quality_threshold = 0.70

y_true = (
    novelty["enhanced_dice"] < quality_threshold
).astype(int)

# Error probability = 1 - reliability
error_score = 1.0 - novelty["reliability"]

fpr, tpr, _ = roc_curve(
    y_true,
    error_score
)

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 6))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"AUROC = {roc_auc:.4f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Reliability-Based Error Detection")

plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path2 = FIG / "Fig2_Error_Detection_ROC.png"
plt.savefig(path2, dpi=600, bbox_inches="tight")
plt.show()

print("✓ Figure 2:", path2)
print(f"AUROC = {roc_auc:.4f}")


# ============================================================
# FIGURE 3 — RISK-COVERAGE CURVE
# ============================================================

thresholds = np.arange(
    0.30,
    0.96,
    0.05
)

coverage = []
risk = []

for t in thresholds:

    accepted = novelty[
        novelty["reliability"] >= t
    ]

    if len(accepted) == 0:
        coverage.append(0)
        risk.append(0)
        continue

    cov = len(accepted) / len(novelty)

    # Risk = segmentation error
    r = 1.0 - accepted["enhanced_dice"].mean()

    coverage.append(cov)
    risk.append(r)

plt.figure(figsize=(7, 5))

plt.plot(
    coverage,
    risk,
    marker="o",
    linewidth=2
)

plt.xlabel("Coverage")
plt.ylabel("Risk (1 − Dice)")
plt.title("Risk–Coverage Curve")

plt.grid(True, alpha=0.25)
plt.tight_layout()

path3 = FIG / "Fig3_Risk_Coverage.png"
plt.savefig(path3, dpi=600, bbox_inches="tight")
plt.show()

print("✓ Figure 3:", path3)


# ============================================================
# FIGURE 4 — RELIABILITY THRESHOLD vs COVERAGE
# ============================================================

thresholds2 = np.arange(
    0.30,
    0.96,
    0.05
)

cov_values = []
accepted_dice = []

for t in thresholds2:

    accepted = novelty[
        novelty["reliability"] >= t
    ]

    cov_values.append(
        len(accepted) / len(novelty)
    )

    if len(accepted):
        accepted_dice.append(
            accepted["enhanced_dice"].mean()
        )
    else:
        accepted_dice.append(np.nan)

fig, ax1 = plt.subplots(
    figsize=(8, 5)
)

ax1.plot(
    thresholds2,
    cov_values,
    marker="o",
    linewidth=2,
    label="Coverage"
)

ax1.set_xlabel("Reliability threshold")
ax1.set_ylabel("Coverage")

ax2 = ax1.twinx()

ax2.plot(
    thresholds2,
    accepted_dice,
    marker="s",
    linewidth=2,
    label="Accepted Dice"
)

ax2.set_ylabel("Accepted Dice")

plt.title(
    "Reliability Threshold vs Selective Performance"
)

ax1.grid(True, alpha=0.25)

plt.tight_layout()

path4 = FIG / "Fig4_Threshold_vs_Coverage_Dice.png"
plt.savefig(path4, dpi=600, bbox_inches="tight")
plt.show()

print("✓ Figure 4:", path4)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("✓ ALL PAPER FIGURES CREATED")
print("=" * 70)

for p in [
    path1,
    path2,
    path3,
    path4,
    path5
]:
    print(p)

## 9. Generate publication figures and export the evidence package

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid',context='paper')
lesion=results[results.lesion_present==True].copy()
plt.figure(figsize=(9,4.8)); sns.boxplot(data=lesion,x='prompt',y='dice',hue='model'); plt.ylabel('Dice (%)'); plt.xlabel('Prompt protocol'); plt.xticks(rotation=25); plt.tight_layout()
plt.savefig(STUDY_ROOT/'figures'/'dice_by_prompt.png',dpi=600,bbox_inches='tight'); plt.show()

if 'BUSSAM' in lesion.model.unique():
    q=lesion[lesion.model=='BUSSAM']; degradation=q.groupby(['seed','prompt']).dice.mean().unstack()
    if 'center' in degradation: degradation=degradation.subtract(degradation.center,axis=0)
    degradation.to_csv(STUDY_ROOT/'tables'/'prompt_dice_change_from_center.csv')

readme=STUDY_ROOT/'README_RESULTS.txt'
readme.write_text(f'''MICA 2026 BUSSAM robustness evidence package
Generated UTC: {pd.Timestamp.utcnow().isoformat()}
Seeds: {SEEDS}
Per-case files: {len(files)}
Important: BUSSAM is an existing published method. These outputs support only reproduction/robustness claims.
Normal/empty-mask cases must be reported separately from lesion-present cases.
Hausdorff values are pixels after 256x256 resizing unless physical spacing is supplied.
''')
archive=shutil.make_archive('/content/MICA2026_BUSSAM_EVIDENCE','zip',STUDY_ROOT)
print('Archive:',archive)
print('Persistent results:',STUDY_ROOT)


In [ ]:
# Purpose: Project lo generate chesina ALL figures + tables + README ni oka ZIP evidence package ga save cheyyadaniki.

import shutil
from pathlib import Path

# Make sure folders exist
(STUDY_ROOT / "figures").mkdir(
    parents=True,
    exist_ok=True
)

(STUDY_ROOT / "tables").mkdir(
    parents=True,
    exist_ok=True
)

# Create ZIP package
archive_path = shutil.make_archive(
    "/kaggle/working/MICA2026_BUSSAM_FINAL_EVIDENCE",
    "zip",
    STUDY_ROOT
)

print("=" * 70)
print("✓ FINAL PROJECT EVIDENCE SAVED")
print("=" * 70)

print("\nZIP FILE:")
print(archive_path)

print("\nFIGURES:")
for f in sorted((STUDY_ROOT / "figures").glob("*")):
    print("✓", f.name)

print("\nTABLES:")
for f in sorted((STUDY_ROOT / "tables").glob("*.csv")):
    print("✓", f.name)

print("\nREADME:")
print(
    STUDY_ROOT / "README_RESULTS.txt"
)

**ekada nunchi piki patuku vellu**

## Required reporting rules

1. Describe BUSSAM as an existing method and cite Tu et al.
2. State that BUSSAM point prompts are derived from reference masks and distinguish oracle/interactively prompted segmentation from automatic segmentation.
3. Report lesion-present and empty-mask cases separately.
4. Treat Hausdorff distance as pixels unless physical pixel spacing is available.
5. Do not select the best seed for the primary result; report all seeds and their mean/dispersion.
6. Keep all failed runs and protocol deviations in the execution log.
